In [1]:
import pandas as pd
import numpy as np
import os, json, re

RAW = '/mnt/shared-workspace/shared/osdr_raw'
PROC = '/mnt/results/data'
os.makedirs(PROC, exist_ok=True)

# ── 1. Parse OSD-575 Comprehensive Metabolic Panel (mineral measurements) ──
cmp_sub = pd.read_csv(f'{RAW}/OSD-575_LSDS-8_Comprehensive_Metabolic_Panel_CMP.upload_SUBMITTED.csv')
print('CMP columns:', list(cmp_sub.columns))
print('CMP analytes:', cmp_sub['ANALYTE'].unique().tolist())
print('CMP subjects:', cmp_sub['SUBJECT_ID'].unique().tolist())
print('CMP timepoints:', sorted(cmp_sub['timepoint'].unique().tolist()))
print('CMP shape:', cmp_sub.shape)

# Pivot to wide format: rows = subject+timepoint, cols = analytes
cmp_wide = cmp_sub.pivot_table(index=['SUBJECT_ID','timepoint'], columns='ANALYTE', values='VALUE', aggfunc='first').reset_index()
cmp_wide.columns.name = None
print('\nCMP wide shape:', cmp_wide.shape)
print(cmp_wide.head())

# Save processed mineral data
minerals_available = ['CALCIUM','POTASSIUM','SODIUM','CHLORIDE','CARBON DIOXIDE']
mineral_df = cmp_wide[['SUBJECT_ID','timepoint'] + [c for c in minerals_available if c in cmp_wide.columns]]
mineral_df.to_csv(f'{PROC}/astronaut_minerals_cmp.csv', index=False)
print('\nSaved astronaut_minerals_cmp.csv')
print(mineral_df)

CMP columns: ['ANALYTE', 'VALUE', 'RANGE_MIN', 'RANGE_MAX', 'UNITS', 'TEST_TYPE', 'SUBJECT_ID', 'SEX', 'timepoint']
CMP analytes: ['GLUCOSE', 'UREA NITROGEN (BUN)', 'CREATININE', 'eGFR NON-AFR. AMERICAN', 'eGFR AFRICAN AMERICAN', 'BUN/CREATININE RATIO', 'SODIUM', 'POTASSIUM', 'CHLORIDE', 'CARBON DIOXIDE', 'CALCIUM', 'PROTEIN; TOTAL', 'ALBUMIN', 'GLOBULIN', 'ALBUMIN/GLOBULIN RATIO', 'BILIRUBIN; TOTAL', 'ALKALINE PHOSPHATASE', 'AST', 'ALT']
CMP subjects: ['C001', 'C002', 'C004', 'C003']
CMP timepoints: ['L-3', 'L-44', 'L-92', 'R+1', 'R+194', 'R+45', 'R+82']
CMP shape: (531, 9)

CMP wide shape: (28, 21)
  SUBJECT_ID timepoint  ALBUMIN  ALBUMIN/GLOBULIN RATIO  ALKALINE PHOSPHATASE  \
0       C001       L-3      4.9                     2.0                  45.0   
1       C001      L-44      5.0                     1.9                  46.0   
2       C001      L-92      5.0                     2.2                  51.0   
3       C001       R+1      4.9                     1.9             


Saved astronaut_minerals_cmp.csv
   SUBJECT_ID timepoint  CALCIUM  POTASSIUM  SODIUM  CHLORIDE  CARBON DIOXIDE
0        C001       L-3      9.6        3.9   142.0     102.0            26.0
1        C001      L-44     10.0        4.2   139.0     101.0            26.0
2        C001      L-92      9.7        4.6   141.0     104.0            23.0
3        C001       R+1      9.7        4.4   138.0     102.0            24.0
4        C001     R+194      9.4        4.3   137.0     102.0            23.0
5        C001      R+45      9.6        4.0   138.0     100.0            28.0
6        C001      R+82     10.1        4.3   140.0     101.0            27.0
7        C002       L-3      9.3        3.8   143.0     106.0            24.0
8        C002      L-44      9.7        4.0   141.0     103.0            26.0
9        C002      L-92     10.2        3.9   142.0     103.0            23.0
10       C002       R+1      9.2        3.5   137.0     103.0            22.0
11       C002     R+194      9

In [2]:
# ── 2. Parse OSD-569 RNA-seq gene expression ──
# The file is an XLSX with processed gene expression
import openpyxl

rna_file = f'{RAW}/OSD-569_GLDS-561_long-readRNAseq_Direct_RNA_seq_Gene_Expression_Processed.xlsx'
wb = openpyxl.load_workbook(rna_file, read_only=True, data_only=True)
print('Sheet names:', wb.sheetnames)
for sn in wb.sheetnames[:3]:
    ws = wb[sn]
    rows = list(ws.iter_rows(max_row=5, values_only=True))
    print(f'\n--- Sheet: {sn} ({ws.max_row} rows x {ws.max_column} cols) ---')
    for r in rows:
        print([str(c)[:30] if c else '' for c in r[:8]])
wb.close()

Sheet names: ['I4-FP1', 'I4-FP2', 'I4-FP3', 'I4-RP1', 'I4-RP2', 'I4-RP4', 'I4-RP5', 'I4-RP6', 'I4-LP1', 'I4-LP2', 'I4-LP4', 'I4-LP5', 'I4-LP6']

--- Sheet: I4-FP1 (61862 rows x 39 cols) ---
['### Mission', 'Inspiration4', '', '', '', '', '', '']
['### Data', 'Direct RNA-seq, Gene Expressio', '', '', '', '', '', '']
['### Database Identifier', 'I4-FP1', '', '', '', '', '', '']
['### Comparison', '(R+1) vs (L-92, L-44, L-3)', '', '', '', '', '', '']
['### Data Note 1', 'FeatureCounts was used to calc', '', '', '', '', '', '']

--- Sheet: I4-FP2 (61862 rows x 55 cols) ---
['### Mission', 'Inspiration4', '', '', '', '', '', '']
['### Data', 'Direct RNA-seq', '', '', '', '', '', '']
['### Database Identifier', 'I4-FP2', '', '', '', '', '', '']
['### Comparison', '(R+1, R+45, R+82) vs (L-92, L-', '', '', '', '', '', '']
['### Data Note 1', 'FeatureCounts was used to calc', '', '', '', '', '', '']

--- Sheet: I4-FP3 (61862 rows x 63 cols) ---
['### Mission', 'Inspiration4', '', '', '', '', ''

In [3]:
# Parse the RNA-seq DE data from OSD-569
# Each sheet has metadata rows starting with ###, then a header row, then data

def parse_osdr_xlsx(filepath, max_sheets=None):
    """Parse OSDR processed XLSX with ### metadata headers."""
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    sheets = wb.sheetnames[:max_sheets] if max_sheets else wb.sheetnames
    all_de = []
    
    for sn in sheets:
        ws = wb[sn]
        rows = list(ws.iter_rows(values_only=True))
        # Find the header row (first row not starting with ###)
        header_idx = None
        for i, r in enumerate(rows):
            if r[0] and not str(r[0]).startswith('###'):
                header_idx = i
                break
        if header_idx is None:
            continue
        headers = [str(c) if c else f'col_{j}' for j, c in enumerate(rows[header_idx])]
        # Extract comparison info from metadata
        comparison = ''
        for r in rows[:header_idx]:
            if r[0] and 'Comparison' in str(r[0]):
                comparison = str(r[1]) if r[1] else ''
                break
        
        # Parse data rows
        for r in rows[header_idx+1:]:
            if not r[0]: continue
            row_dict = {headers[j]: r[j] for j in range(min(len(headers), len(r)))}
            row_dict['_sheet'] = sn
            row_dict['_comparison'] = comparison
            all_de.append(row_dict)
    
    wb.close()
    return pd.DataFrame(all_de)

rna_de = parse_osdr_xlsx(rna_file, max_sheets=3)
print('RNA-seq DE shape:', rna_de.shape)
print('Columns:', [c for c in rna_de.columns if not c.startswith('_')][:15])
print('\nFirst few rows:')
print(rna_de[['Gene_name','Gene_id','log2FoldChange','padj','_sheet','_comparison']].head(10) 
      if 'Gene_name' in rna_de.columns else rna_de.iloc[:5, :8])

RNA-seq DE shape: (185556, 65)
Columns: ['ENSEMBL', 'col_1', 'col_2', 'col_3', 'col_4', 'col_5', 'col_6', 'col_7', 'col_8', 'col_9', 'col_10', 'col_11', 'col_12', 'col_13', 'col_14']

First few rows:
              ENSEMBL  col_1  col_2  col_3  col_4  col_5  col_6  col_7
0  ENSG00000000003.15      1      0      0      0      0      0      1
1   ENSG00000000005.6      0      0      0      0      0      0      0
2  ENSG00000000419.14      5      9      8      7      2      3     36
3  ENSG00000000457.14      2      0      2      0      2      1      7
4  ENSG00000000460.17      0      0      0      0      0      0      1


In [4]:
# Check the actual header row of the first sheet
wb = openpyxl.load_workbook(rna_file, read_only=True, data_only=True)
ws = wb['I4-FP1']
rows = list(ws.iter_rows(max_row=20, values_only=True))
for i, r in enumerate(rows):
    vals = [str(c)[:25] if c else '' for c in r[:12]]
    print(f'Row {i}: {vals}')
wb.close()

Row 0: ['### Mission', 'Inspiration4', '', '', '', '', '', '', '', '', '', '']
Row 1: ['### Data', 'Direct RNA-seq, Gene Expr', '', '', '', '', '', '', '', '', '', '']
Row 2: ['### Database Identifier', 'I4-FP1', '', '', '', '', '', '', '', '', '', '']
Row 3: ['### Comparison', '(R+1) vs (L-92, L-44, L-3', '', '', '', '', '', '', '', '', '', '']
Row 4: ['### Data Note 1', 'FeatureCounts was used to', '', '', '', '', '', '', '', '', '', '']
Row 5: ['### Data Note 2', 'Positive logFC = higher a', '', '', '', '', '', '', '', '', '', '']
Row 6: ['', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts']
Row 7: ['', 'C001', 'C001', 'C001', 'C001', 'C002', 'C002', 'C002', 'C002', 'C003', 'C003', 'C003']
Row 8: ['', 'L-92', 'L-44', 'L-3', 'R+1', 'L-92', 'L-44', 'L-3', 'R+1', 'L-92', 'L-44', 'L-3']
Row 9: ['ENSEMBL', '', '', '', '', '', '', '', '', '', '', '']
Row

In [5]:
# Check all columns of first sheet
wb = openpyxl.load_workbook(rna_file, read_only=True, data_only=True)
ws = wb['I4-FP1']
# Get header rows 6-9 and a data row
for ridx in [6,7,8,9]:
    r = list(ws.iter_rows(min_row=ridx+1, max_row=ridx+1, values_only=True))[0]
    print(f'Row {ridx}: {[str(c)[:20] if c else "." for c in r]}')
print(f'\nTotal columns: {ws.max_column}')
# Check last columns for DE results
r = list(ws.iter_rows(min_row=10, max_row=10, values_only=True))[0]
print(f'Data row 10 (last 15 cols): {[str(c)[:20] if c else "." for c in r[-15:]]}')
# Check header row 9 for last columns
r9 = list(ws.iter_rows(min_row=10, max_row=10, values_only=True))[0]
print(f'Header row 9 (last 15 cols): {[str(c)[:20] if c else "." for c in r9[-15:]]}')
wb.close()

Row 6: ['.', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'featureCounts', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'salmon', 'I4-FP1', 'I4-FP1', 'I4-FP1', 'I4-FP1', 'I4-FP1', 'I4-FP1']
Row 7: ['.', 'C001', 'C001', 'C001', 'C001', 'C002', 'C002', 'C002', 'C002', 'C003', 'C003', 'C003', 'C003', 'C004', 'C004', 'C004', 'C004', 'C001', 'C001', 'C001', 'C001', 'C002', 'C002', 'C002', 'C002', 'C003', 'C003', 'C003', 'C003', 'C004', 'C004', 'C004', 'C004', 'DESeq2', 'DESeq2', 'DESeq2', 'pipeline-transcripto', 'pipeline-transcripto', 'pipeline-transcripto']
Row 8: ['.', 'L-92', 'L-44', 'L-3', 'R+1', 'L-92', 'L-44', 'L-3', 'R+1', 'L-92', 'L-44', 'L-3', 'R+1', 'L-92', 'L-44', 'L-3'

In [10]:
# Fixed parser: DE method names are in subject_row, not source_row
def safe_float(val):
    try: return float(val)
    except (ValueError, TypeError): return 0.0

def parse_osdr_rnaseq(filepath, sheets=None):
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    sheets = sheets or wb.sheetnames
    all_de = []
    count_matrices = {}
    
    for sn in sheets:
        ws = wb[sn]
        rows = []
        for i, r in enumerate(ws.iter_rows(values_only=True)):
            if i > 65000: break
            rows.append(r)
        
        meta = {}
        header_start = None
        for i, r in enumerate(rows):
            if r[0] and str(r[0]).startswith('###'):
                meta[str(r[0]).replace('### ','')] = str(r[1]) if r[1] else ''
            if any(r[j] and 'featureCounts' in str(r[j]) for j in range(1, min(len(r), 20))):
                header_start = i
                break
        if header_start is None: continue
        
        source_row = rows[header_start]
        subject_row = rows[header_start + 1] if header_start+1 < len(rows) else []
        timepoint_row = rows[header_start + 2] if header_start+2 < len(rows) else []
        
        col_names = ['ENSEMBL']
        for j in range(1, len(source_row)):
            src = str(source_row[j]) if source_row[j] else ''
            subj = str(subject_row[j]) if j < len(subject_row) and subject_row[j] else ''
            tp = str(timepoint_row[j]) if j < len(timepoint_row) and timepoint_row[j] else ''
            if src == 'featureCounts' and subj and tp:
                col_names.append(f'{subj}_{tp}')
            elif src == 'salmon' and subj and tp:
                col_names.append(f'{subj}_{tp}_tpm')
            elif subj in ('DESeq2', 'pipeline-transcriptome-de'):
                col_names.append(f'{subj}_{tp}')
            else:
                col_names.append(f'col_{j}')
        
        data_start = header_start + 4
        de_cols = [j for j, cn in enumerate(col_names) if cn.startswith('DESeq2_') or cn.startswith('pipeline-transcriptome-de_')]
        count_cols = [j for j, cn in enumerate(col_names) if cn.endswith(('_L-92','_L-44','_L-3','_R+1','_R+45','_R+82','_R+194')) and '_tpm' not in cn]
        
        sheet_de = []
        sheet_counts = []
        for r in rows[data_start:]:
            if not r[0]: continue
            gene_id = str(r[0])
            de_row = {'ENSEMBL': gene_id, '_sheet': sn, '_comparison': meta.get('Comparison','')}
            for j in de_cols:
                if j < len(r) and j < len(col_names):
                    de_row[col_names[j]] = safe_float(r[j])
            sheet_de.append(de_row)
            count_row = {'ENSEMBL': gene_id}
            for j in count_cols:
                if j < len(r) and j < len(col_names):
                    count_row[col_names[j]] = safe_float(r[j])
            sheet_counts.append(count_row)
        
        all_de.extend(sheet_de)
        if sheet_counts:
            count_matrices[sn] = pd.DataFrame(sheet_counts)
    
    wb.close()
    return pd.DataFrame(all_de), count_matrices

rna_de, rna_counts = parse_osdr_rnaseq(rna_file, sheets=['I4-FP1','I4-FP2','I4-FP3'])
print('DE results shape:', rna_de.shape)
de_cols = [c for c in rna_de.columns if c.startswith('DESeq2_') or c.startswith('pipeline_')]
print('DE columns:', de_cols)
print('\nCount matrices:', {k: v.shape for k, v in rna_counts.items()})
print('\nSample DE results (I4-FP1, top by adj p):')
fp1_de = rna_de[rna_de['_sheet']=='I4-FP1'].copy()
if 'DESeq2_adjusted p-value' in fp1_de.columns:
    fp1_de = fp1_de.sort_values('DESeq2_adjusted p-value')
    print(fp1_de[['ENSEMBL'] + de_cols].head(10).to_string())
else:
    print('Columns:', list(fp1_de.columns))

# Save
rna_de.to_csv(f'{PROC}/astronaut_rnaseq_de.csv', index=False)
if 'I4-FP1' in rna_counts:
    rna_counts['I4-FP1'].to_csv(f'{PROC}/astronaut_rnaseq_counts.csv', index=False)
    print(f'\nSaved counts: {rna_counts["I4-FP1"].shape}')
print(f'Saved DE: {rna_de.shape}')

DE results shape: (185556, 9)
DE columns: ['DESeq2_log2FC', 'DESeq2_p-value', 'DESeq2_adjusted p-value']

Count matrices: {'I4-FP1': (61852, 17), 'I4-FP2': (61852, 25), 'I4-FP3': (61852, 29)}

Sample DE results (I4-FP1, top by adj p):
                 ENSEMBL  DESeq2_log2FC  DESeq2_p-value  DESeq2_adjusted p-value
41221  ENSG00000253574.5            0.0        0.000000                      0.0
41206  ENSG00000253557.6            0.0        0.000000                      0.0
41207  ENSG00000253558.2            0.0        0.000000                      0.0
41208  ENSG00000253559.1            0.0        0.899483                      0.0
41209  ENSG00000253560.1            0.0        0.000000                      0.0
41210  ENSG00000253561.1            0.0        0.000000                      0.0
41211  ENSG00000253562.1            0.0        0.000000                      0.0
41212  ENSG00000253563.2            0.0        0.000000                      0.0
41213  ENSG00000253564.1            


Saved counts: (61852, 17)
Saved DE: (185556, 9)


In [7]:
# Debug: print actual column names from I4-FP1
cm = rna_counts['I4-FP1']
print('All column names:')
for i, c in enumerate(cm.columns):
    print(f'  {i}: {c}')

All column names:
  0: ENSEMBL
  1: col_1
  2: col_2
  3: col_3
  4: col_4
  5: col_5
  6: col_6
  7: col_7
  8: col_8
  9: col_9
  10: col_10
  11: col_11
  12: col_12
  13: col_13
  14: col_14
  15: col_15
  16: col_16
  17: col_17
  18: col_18
  19: col_19
  20: col_20
  21: col_21
  22: col_22
  23: col_23
  24: col_24
  25: col_25
  26: col_26
  27: col_27
  28: col_28
  29: col_29
  30: col_30
  31: col_31
  32: col_32
  33: col_33
  34: 0.869038955318808_0.401303953631694
  35: NA_NA
  36: 0.0477397104469443_0.555946295739047
  37: 0.899644666021314_0.356439722343672
  38: 0.989713755382559_0.932855093968936


In [9]:
# Debug: check col_names for DE columns
wb = openpyxl.load_workbook(rna_file, read_only=True, data_only=True)
ws = wb['I4-FP1']
rows = []
for i, r in enumerate(ws.iter_rows(values_only=True)):
    if i > 15: break
    rows.append(r)

# Find header_start
for i, r in enumerate(rows):
    if any(r[j] and 'featureCounts' in str(r[j]) for j in range(1, min(len(r), 20))):
        print(f'header_start = {i}')
        source_row = r
        subject_row = rows[i+1]
        timepoint_row = rows[i+2]
        break

print(f'\nSource row (cols 30-38): {[str(source_row[j])[:25] if j < len(source_row) and source_row[j] else "." for j in range(30, 39)]}')
print(f'Subject row (cols 30-38): {[str(subject_row[j])[:25] if j < len(subject_row) and subject_row[j] else "." for j in range(30, 39)]}')
print(f'Timepoint row (cols 30-38): {[str(timepoint_row[j])[:25] if j < len(timepoint_row) and timepoint_row[j] else "." for j in range(30, 39)]}')

# Build col_names for DE columns
for j in range(33, 39):
    src = str(source_row[j]) if j < len(source_row) and source_row[j] else ''
    subj = str(subject_row[j]) if j < len(subject_row) and subject_row[j] else ''
    tp = str(timepoint_row[j]) if j < len(timepoint_row) and timepoint_row[j] else ''
    print(f'  col {j}: src="{src}" subj="{subj}" tp="{tp}" -> name would be: ', end='')
    if src == 'DESeq2':
        print(f'DESeq2_{tp}')
    elif src == 'pipeline-transcriptome-de':
        print(f'pipeline_{tp}')
    else:
        print(f'col_{j} (src not recognized)')
wb.close()

header_start = 6

Source row (cols 30-38): ['salmon', 'salmon', 'salmon', 'I4-FP1', 'I4-FP1', 'I4-FP1', 'I4-FP1', 'I4-FP1', 'I4-FP1']
Subject row (cols 30-38): ['C004', 'C004', 'C004', 'DESeq2', 'DESeq2', 'DESeq2', 'pipeline-transcriptome-de', 'pipeline-transcriptome-de', 'pipeline-transcriptome-de']
Timepoint row (cols 30-38): ['L-44', 'L-3', 'R+1', 'log2FC', 'p-value', 'adjusted p-value', 'log2FC', 'p-value', 'adjusted p-value']
  col 33: src="I4-FP1" subj="DESeq2" tp="log2FC" -> name would be: col_33 (src not recognized)
  col 34: src="I4-FP1" subj="DESeq2" tp="p-value" -> name would be: col_34 (src not recognized)
  col 35: src="I4-FP1" subj="DESeq2" tp="adjusted p-value" -> name would be: col_35 (src not recognized)
  col 36: src="I4-FP1" subj="pipeline-transcriptome-de" tp="log2FC" -> name would be: col_36 (src not recognized)
  col 37: src="I4-FP1" subj="pipeline-transcriptome-de" tp="p-value" -> name would be: col_37 (src not recognized)
  col 38: src="I4-FP1" subj="pipeline-tr

In [11]:
# ── 3. Parse OSD-571 proteomics + metabolomics ──
prot_file = f'{RAW}/OSD-571_GLDS-563_proteomics_Plasma_Proteomics_Processed_Data.xlsx'
metab_file = f'{RAW}/OSD-571_GLDS-563_metabolomics_Plasma_Metabolomics_Processed_Data.xlsx'

# Proteomics
wb = openpyxl.load_workbook(prot_file, read_only=True, data_only=True)
print('Proteomics sheets:', wb.sheetnames)
for sn in wb.sheetnames[:2]:
    ws = wb[sn]
    rows = list(ws.iter_rows(max_row=8, values_only=True))
    print(f'\n--- {sn} ({ws.max_row}x{ws.max_column}) ---')
    for r in rows:
        print([str(c)[:25] if c else '.' for c in r[:8]])
wb.close()

print('\n\n=== METABOLOMICS ===')
wb = openpyxl.load_workbook(metab_file, read_only=True, data_only=True)
print('Metabolomics sheets:', wb.sheetnames)
for sn in wb.sheetnames[:2]:
    ws = wb[sn]
    rows = list(ws.iter_rows(max_row=8, values_only=True))
    print(f'\n--- {sn} ({ws.max_row}x{ws.max_column}) ---')
    for r in rows:
        print([str(c)[:25] if c else '.' for c in r[:8]])
wb.close()

Proteomics sheets: ['I4-FP1', 'I4-FP2']

--- I4-FP1 (1772x7) ---
['### Mission', 'Inspiration4', '.', '.', '.', '.', '.']
['### Data', 'Plasma Proteomics', '.', '.', '.', '.', '.']
['### Database Identifier', 'I4-FP1', '.', '.', '.', '.', '.']
['### Comparison', '(R+1) vs (L-92, L-44, L-3', '.', '.', '.', '.', '.']
['### Data Note 1', 'Positive logFC = higher a', '.', '.', '.', '.', '.']
['### Data Note 2', 'Calculated with limma (ve', '.', '.', '.', '.', '.']
['Gene', 'logFC', 'AveExpr', 't', 'P.Value', 'adj.P.Val', 'B']
['PF4', '1.44567063619363', '18.6302855610587', '7.95798103347458', '2.58994739825387e-07', '0.000457125715791809', '7.04396592906646']

--- I4-FP2 (1527x7) ---
['### Mission', 'Inspiration4', '.', '.', '.', '.', '.']
['### Data', 'Plasma Proteomics', '.', '.', '.', '.', '.']
['### Database Identifier', 'I4-FP2', '.', '.', '.', '.', '.']
['### Comparison', '(R+1, R+45, R+82) vs (L-9', '.', '.', '.', '.', '.']
['### Data Note 1', 'Positive logFC = higher a', '.', '.', 

In [12]:
# Parse proteomics, metabolomics, and CBC data
def parse_simple_de_xlsx(filepath):
    """Parse OSDR XLSX with ### metadata + simple header row."""
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    all_de = []
    for sn in wb.sheetnames:
        ws = wb[sn]
        rows = list(ws.iter_rows(values_only=True))
        meta = {}
        header_idx = None
        for i, r in enumerate(rows):
            if r[0] and str(r[0]).startswith('###'):
                meta[str(r[0]).replace('### ','')] = str(r[1]) if r[1] else ''
            elif r[0] and not str(r[0]).startswith('###') and i > 3:
                header_idx = i
                break
        if header_idx is None: continue
        headers = [str(c) if c else f'col_{j}' for j, c in enumerate(rows[header_idx])]
        for r in rows[header_idx+1:]:
            if not r[0]: continue
            row = {headers[j]: safe_float(r[j]) if j > 0 and j < len(r) else (r[j] if j < len(r) else None) for j in range(len(headers))}
            row['_sheet'] = sn
            row['_comparison'] = meta.get('Comparison','')
            # Keep the first column as string (gene/metabolite ID)
            row[headers[0]] = str(r[0])
            all_de.append(row)
    wb.close()
    return pd.DataFrame(all_de)

# Proteomics
prot_de = parse_simple_de_xlsx(prot_file)
print('Proteomics DE:', prot_de.shape)
print(prot_de[['Gene','logFC','adj.P.Val','_sheet']].head(10))
prot_de.to_csv(f'{PROC}/astronaut_proteomics_de.csv', index=False)

# Metabolomics
metab_de = parse_simple_de_xlsx(metab_file)
print('\nMetabolomics DE:', metab_de.shape)
print(metab_de[['ID','logFC','adj.P.Val','_sheet']].head(10))
metab_de.to_csv(f'{PROC}/astronaut_metabolomics_de.csv', index=False)

# CBC data (hemoglobin = iron proxy)
cbc = pd.read_csv(f'{RAW}/OSD-569_LSDS-7_Complete_Blood_Count_CBC.upload_SUBMITTED.csv')
print('\nCBC shape:', cbc.shape)
print('CBC analytes:', cbc['ANALYTE'].unique().tolist())
# Pivot CBC
cbc_wide = cbc.pivot_table(index=['SUBJECT_ID','timepoint'], columns='ANALYTE', values='VALUE', aggfunc='first').reset_index()
cbc_wide.columns.name = None
print('CBC wide:', cbc_wide.shape)
# Hemoglobin as iron proxy
iron_proxy = cbc_wide[['SUBJECT_ID','timepoint','HEMOGLOBIN','HEMATOCRIT','RED BLOOD CELL COUNT']].copy() if 'HEMOGLOBIN' in cbc_wide.columns else cbc_wide[['SUBJECT_ID','timepoint']].copy()
iron_proxy.to_csv(f'{PROC}/astronaut_cbc_iron_proxy.csv', index=False)
print('\nIron proxy (hemoglobin):')
print(iron_proxy.head(10))

Proteomics DE: (3285, 9)
    Gene     logFC  adj.P.Val  _sheet
0    PF4  1.445671   0.000457  I4-FP1
1   QPCT -1.604759   0.001045  I4-FP1
2  TGFB1  1.483192   0.001045  I4-FP1
3    APP  1.320346   0.001148  I4-FP1
4   CCN2  1.277208   0.001868  I4-FP1
5    A2M  1.131542   0.002183  I4-FP1
6  LTBP1  1.307592   0.002253  I4-FP1
7  PF4V1  1.755976   0.002534  I4-FP1
8   SNCA  1.793406   0.002570  I4-FP1
9    LPL -1.437323   0.002858  I4-FP1



Metabolomics DE: (2624, 11)
                        ID     logFC  adj.P.Val  _sheet
0             LysoPC(15:0) -2.341561   0.000037  I4-FP1
1             LysoPC(14:0) -2.108057   0.000217  I4-FP1
2             LysoPC(17:0) -2.046564   0.000217  I4-FP1
3                  Inosine  2.160162   0.000245  I4-FP1
4  Sphingosine 1-phosphate -1.907023   0.000585  I4-FP1
5             LysoPC(16:1) -1.884126   0.000703  I4-FP1
6                PC(P38:3)  1.806443   0.000911  I4-FP1
7         FA (26:0) [M-H]-  1.902006   0.000911  I4-FP1
8             LysoPC(18:0) -1.708873   0.001232  I4-FP1
9             LysoPC(16:0) -1.841156   0.001756  I4-FP1



CBC shape: (553, 9)
CBC analytes: ['WHITE BLOOD CELL COUNT', 'RED BLOOD CELL COUNT', 'HEMOGLOBIN', 'HEMATOCRIT', 'MCV', 'MCH', 'MCHC', 'RDW', 'PLATELET COUNT', 'MPV', 'ABSOLUTE NEUTROPHILS', 'ABSOLUTE LYMPHOCYTES', 'ABSOLUTE MONOCYTES', 'ABSOLUTE EOSINOPHILS', 'ABSOLUTE BASOPHILS', 'NEUTROPHILS', 'LYMPHOCYTES', 'MONOCYTES', 'EOSINOPHILS', 'BASOPHILS', 'RED BLOOD CELL COUNT (FEMALE)', 'HEMOGLOBIN (FEMALE)', 'HEMATOCRIT (FEMALE)']


KeyError: 'timepoint'

In [13]:
# Fix CBC parsing - check columns
cbc = pd.read_csv(f'{RAW}/OSD-569_LSDS-7_Complete_Blood_Count_CBC.upload_SUBMITTED.csv')
print('CBC columns:', list(cbc.columns))
print('CBC TEST_DATE values:', sorted(cbc['TEST_DATE'].astype(str).unique())[:15])
print('CBC head:')
print(cbc.head(5))

# The timepoint might be in TEST_DATE or we need to extract from sample
# Check if there's a timepoint pattern in TEST_DATE
print('\nUnique TEST_DATE:', cbc['TEST_DATE'].unique()[:20])

CBC columns: ['ANALYTE', 'VALUE', 'RANGE_MIN', 'RANGE_MAX', 'UNITS', 'TEST_TYPE', 'SUBJECT_ID', 'SEX', 'TEST_DATE']
CBC TEST_DATE values: ['L-3', 'L-44', 'L-92', 'R+1', 'R+194', 'R+45', 'R+82']
CBC head:
                  ANALYTE  VALUE  RANGE_MIN  RANGE_MAX        UNITS TEST_TYPE  \
0  WHITE BLOOD CELL COUNT   5.40        3.8       10.8  Thousand/uL       CBC   
1    RED BLOOD CELL COUNT   4.58        4.2        5.8   Million/uL       CBC   
2              HEMOGLOBIN  14.10       13.2       17.1         g/dL       CBC   
3              HEMATOCRIT  42.50       38.5       50.0            %       CBC   
4                     MCV  92.80       80.0      100.0           fL       CBC   

  SUBJECT_ID SEX TEST_DATE  
0       C001   M      L-92  
1       C001   M      L-92  
2       C001   M      L-92  
3       C001   M      L-92  
4       C001   M      L-92  

Unique TEST_DATE: ['L-92' 'L-44' 'L-3' 'R+1' 'R+45' 'R+82' 'R+194']


In [14]:
# Fix CBC pivot using TEST_DATE as timepoint
cbc_wide = cbc.pivot_table(index=['SUBJECT_ID','TEST_DATE'], columns='ANALYTE', values='VALUE', aggfunc='first').reset_index()
cbc_wide.columns.name = None
cbc_wide = cbc_wide.rename(columns={'TEST_DATE':'timepoint'})
print('CBC wide:', cbc_wide.shape)
print(cbc_wide.head())

# Iron proxy: hemoglobin, hematocrit, RBC count
iron_cols = [c for c in ['HEMOGLOBIN','HEMATOCRIT','RED BLOOD CELL COUNT','MCH','MCHC','MCV'] if c in cbc_wide.columns]
iron_proxy = cbc_wide[['SUBJECT_ID','timepoint'] + iron_cols].copy()
iron_proxy.to_csv(f'{PROC}/astronaut_cbc_iron_proxy.csv', index=False)
print(f'\nIron proxy saved: {iron_proxy.shape}')
print(iron_proxy.head(10))

# Save full CBC
cbc_wide.to_csv(f'{PROC}/astronaut_cbc_full.csv', index=False)

# Combine mineral data: CMP + CBC iron proxy
mineral_combined = mineral_df.merge(iron_proxy, on=['SUBJECT_ID','timepoint'], how='outer')
mineral_combined.to_csv(f'{PROC}/astronaut_minerals_combined.csv', index=False)
print(f'\nCombined mineral data: {mineral_combined.shape}')
print(mineral_combined.head(10))

CBC wide: (28, 25)
  SUBJECT_ID timepoint  ABSOLUTE BASOPHILS  ABSOLUTE EOSINOPHILS  \
0       C001       L-3                49.0                 161.0   
1       C001      L-44                50.0                 204.0   
2       C001      L-92                70.0                 259.0   
3       C001       R+1                50.0                 170.0   
4       C001     R+194                49.0                 221.0   

   ABSOLUTE LYMPHOCYTES  ABSOLUTE MONOCYTES  ABSOLUTE NEUTROPHILS  BASOPHILS  \
0                2667.0               553.0                3570.0        0.7   
1                1859.0               391.0                2998.0        0.9   
2                2241.0               481.0                2349.0        1.3   
3                1640.0               455.0                2685.0        1.0   
4                1847.0               392.0                2391.0        1.0   

   EOSINOPHILS  HEMATOCRIT  ...  MCHC   MCV  MONOCYTES   MPV  NEUTROPHILS  \
0          2.3


Iron proxy saved: (28, 8)
  SUBJECT_ID timepoint  HEMOGLOBIN  HEMATOCRIT  RED BLOOD CELL COUNT   MCH  \
0       C001       L-3        14.7        44.5                  4.69  31.3   
1       C001      L-44        14.9        45.1                  4.85  30.7   
2       C001      L-92        14.1        42.5                  4.58  30.8   
3       C001       R+1        14.7        43.6                  4.84  30.4   
4       C001     R+194        14.1        41.8                  4.66  30.3   
5       C001      R+45        14.3        43.3                  4.63  30.9   
6       C001      R+82        15.7        46.1                  5.02  31.3   
7       C002       L-3        11.5        35.9                  3.72  30.9   
8       C002      L-44        12.8        37.8                  3.93  32.6   
9       C002      L-92        13.2        40.8                  4.36  30.3   

   MCHC   MCV  
0  33.0  94.9  
1  33.0  93.0  
2  33.2  92.8  
3  33.7  90.1  
4  33.7  89.7  
5  33.0  93.5  
6 


Combined mineral data: (28, 13)
  SUBJECT_ID timepoint  CALCIUM  POTASSIUM  SODIUM  CHLORIDE  CARBON DIOXIDE  \
0       C001       L-3      9.6        3.9   142.0     102.0            26.0   
1       C001      L-44     10.0        4.2   139.0     101.0            26.0   
2       C001      L-92      9.7        4.6   141.0     104.0            23.0   
3       C001       R+1      9.7        4.4   138.0     102.0            24.0   
4       C001     R+194      9.4        4.3   137.0     102.0            23.0   
5       C001      R+45      9.6        4.0   138.0     100.0            28.0   
6       C001      R+82     10.1        4.3   140.0     101.0            27.0   
7       C002       L-3      9.3        3.8   143.0     106.0            24.0   
8       C002      L-44      9.7        4.0   141.0     103.0            26.0   
9       C002      L-92     10.2        3.9   142.0     103.0            23.0   

   HEMOGLOBIN  HEMATOCRIT  RED BLOOD CELL COUNT   MCH  MCHC   MCV  
0        14.7     

In [15]:
# ── Stage 2 continued: ENSEMBL→Symbol mapping + mineral pathway cross-reference ──
import pandas as pd, numpy as np, csv, os

PROC = "/mnt/results/data"

# Load ENSEMBL-to-symbol mapping
print("Loading ENSEMBL-to-symbol mapping...")
ens_map = pd.read_csv(f"{PROC}/ensembl_to_symbol.csv")
ens_dict = dict(zip(ens_map["ensembl_id"], ens_map["gene_symbol"]))
print(f"  {len(ens_dict)} entries, {sum(1 for v in ens_dict.values() if v)} mapped")

# Load mineral-pathway crosswalk
print("Loading mineral-pathway crosswalk...")
crosswalk = pd.read_csv(f"{PROC}/mineral_pathway_crosswalk.csv")
mineral_genes = set(crosswalk["gene_symbol"].dropna())
print(f"  {len(mineral_genes)} unique mineral-pathway genes")

# Load RNA-seq DE data
print("Loading RNA-seq DE data...")
rna_de = pd.read_csv(f"{PROC}/astronaut_rnaseq_de.csv")
print(f"  {len(rna_de)} rows, {rna_de['_sheet'].unique()} sheets")

# Add gene symbols
rna_de["gene_symbol"] = rna_de["ENSEMBL"].map(ens_dict)
rna_de_mapped = rna_de.dropna(subset=["gene_symbol"])
print(f"  {len(rna_de_mapped)} rows with mapped symbols ({100*len(rna_de_mapped)/len(rna_de):.1f}%)")

# Flag mineral-pathway genes
rna_de_mapped["is_mineral_gene"] = rna_de_mapped["gene_symbol"].isin(mineral_genes)
print(f"  {rna_de_mapped['is_mineral_gene'].sum()} rows are mineral-pathway genes")

# Save mapped RNA-seq DE
rna_de_mapped.to_csv(f"{PROC}/astronaut_rnaseq_de_mapped.csv", index=False)
print(f"  Saved mapped RNA-seq DE")

# Show mineral-pathway DE genes per comparison
print("\nMineral-pathway DE genes per comparison (|log2FC| > 1, adj p < 0.05):")
for sheet in rna_de_mapped["_sheet"].unique():
    sub = rna_de_mapped[(rna_de_mapped["_sheet"] == sheet) & (rna_de_mapped["is_mineral_gene"])]
    # Use DESeq2 columns
    sig = sub[(sub["DESeq2_log2FC"].abs() > 1) & (sub["DESeq2_adjusted p-value"] < 0.05)]
    print(f"  {sheet}: {len(sig)} significant mineral-pathway DE genes")
    if len(sig) > 0:
        top = sig.nlargest(5, "DESeq2_log2FC", keep="first")[["gene_symbol", "DESeq2_log2FC", "DESeq2_adjusted p-value"]]
        print(top.to_string(index=False))


Loading ENSEMBL-to-symbol mapping...


  61852 entries, 61852 mapped
Loading mineral-pathway crosswalk...
  801 unique mineral-pathway genes
Loading RNA-seq DE data...


  185556 rows, ['I4-FP1' 'I4-FP2' 'I4-FP3'] sheets
  136650 rows with mapped symbols (73.6%)
  2358 rows are mineral-pathway genes


/tmp/ipykernel_445/1584290501.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rna_de_mapped["is_mineral_gene"] = rna_de_mapped["gene_symbol"].isin(mineral_genes)


  Saved mapped RNA-seq DE

Mineral-pathway DE genes per comparison (|log2FC| > 1, adj p < 0.05):
  I4-FP1: 0 significant mineral-pathway DE genes
  I4-FP2: 0 significant mineral-pathway DE genes
  I4-FP3: 0 significant mineral-pathway DE genes


In [16]:
# Examine DE data distributions and relax thresholds
print("=== DESeq2 log2FC distribution (mineral genes) ===")
mineral_rna = rna_de_mapped[rna_de_mapped["is_mineral_gene"]]
print(mineral_rna["DESeq2_log2FC"].describe())
print(f"\nNon-zero log2FC: {(mineral_rna['DESeq2_log2FC'] != 0).sum()}")
print(f"|log2FC| > 0.5: {(mineral_rna['DESeq2_log2FC'].abs() > 0.5).sum()}")
print(f"|log2FC| > 1: {(mineral_rna['DESeq2_log2FC'].abs() > 1).sum()}")

print("\n=== adj p-value distribution ===")
print(mineral_rna["DESeq2_adjusted p-value"].describe())
print(f"adj p < 0.05: {(mineral_rna['DESeq2_adjusted p-value'] < 0.05).sum()}")
print(f"adj p < 0.1: {(mineral_rna['DESeq2_adjusted p-value'] < 0.1).sum()}")
print(f"adj p == 0: {(mineral_rna['DESeq2_adjusted p-value'] == 0).sum()}")

# Check pipeline-transcriptome-de columns
print("\n=== pipeline-transcriptome-de log2FC ===")
print(mineral_rna["pipeline-transcriptome-de_log2FC"].describe())
print(f"|log2FC| > 0.5: {(mineral_rna['pipeline-transcriptome-de_log2FC'].abs() > 0.5).sum()}")
print(f"|log2FC| > 1: {(mineral_rna['pipeline-transcriptome-de_log2FC'].abs() > 1).sum()}")
print(f"adj p < 0.05: {(mineral_rna['pipeline-transcriptome-de_adjusted p-value'] < 0.05).sum()}")

# Relaxed thresholds: |log2FC| > 0.5, p < 0.05 (raw p, not adj)
print("\n=== Relaxed: |log2FC| > 0.5, raw p < 0.05 ===")
for sheet in mineral_rna["_sheet"].unique():
    sub = mineral_rna[mineral_rna["_sheet"] == sheet]
    sig_desq = sub[(sub["DESeq2_log2FC"].abs() > 0.5) & (sub["DESeq2_p-value"] < 0.05)]
    sig_pipe = sub[(sub["pipeline-transcriptome-de_log2FC"].abs() > 0.5) & (sub["pipeline-transcriptome-de_p-value"] < 0.05)]
    print(f"  {sheet}: DESeq2={len(sig_desq)}, pipeline={len(sig_pipe)}")
    if len(sig_pipe) > 0:
        top = sig_pipe.nlargest(10, "pipeline-transcriptome-de_log2FC")[["gene_symbol", "pipeline-transcriptome-de_log2FC", "pipeline-transcriptome-de_p-value"]]
        print(top.to_string(index=False))


=== DESeq2 log2FC distribution (mineral genes) ===
count    2358.000000
mean       -0.000003
std         0.000140
min        -0.002000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.001000
Name: DESeq2_log2FC, dtype: float64

Non-zero log2FC: 43
|log2FC| > 0.5: 0
|log2FC| > 1: 0

=== adj p-value distribution ===
count    2358.000000
mean        0.491373
std         0.496371
min         0.000000
25%         0.000000
50%         0.000000
75%         0.999942
max         0.999988
Name: DESeq2_adjusted p-value, dtype: float64
adj p < 0.05: 1186
adj p < 0.1: 1186
adj p == 0: 1185

=== pipeline-transcriptome-de log2FC ===
count    2358.000000
mean       -0.044434
std         0.469390
min        -2.911668
25%        -0.170545
50%         0.000000
75%         0.030360
max         2.391452
Name: pipeline-transcriptome-de_log2FC, dtype: float64
|log2FC| > 0.5: 423
|log2FC| > 1: 152
adj p < 0.05: 794

=== Relaxed: |log2FC| > 0.5, raw p < 0.05 ===
  I4-FP1: DESeq2=0, 

In [17]:
# ── Comprehensive mineral-pathway DE summary across omics layers ──
import pandas as pd, numpy as np

PROC = "/mnt/results/data"

# Load crosswalk for mineral assignments
crosswalk = pd.read_csv(f"{PROC}/mineral_pathway_crosswalk.csv")
gene_to_minerals = dict(zip(crosswalk["gene_symbol"], crosswalk["minerals"]))

# ── RNA-seq: use pipeline-transcriptome-de as primary ──
print("=== RNA-seq mineral-pathway DE (pipeline-transcriptome-de) ===")
rna_mapped = pd.read_csv(f"{PROC}/astronaut_rnaseq_de_mapped.csv")
mineral_rna = rna_mapped[rna_mapped["is_mineral_gene"]].copy()

rna_sig_all = []
for sheet in mineral_rna["_sheet"].unique():
    sub = mineral_rna[mineral_rna["_sheet"] == sheet].copy()
    sub["sig"] = (sub["pipeline-transcriptome-de_log2FC"].abs() > 0.5) & (sub["pipeline-transcriptome-de_p-value"] < 0.05)
    sig = sub[sub["sig"]].copy()
    sig["omics"] = "RNA-seq"
    sig["comparison"] = sheet
    sig["log2FC"] = sig["pipeline-transcriptome-de_log2FC"]
    sig["pvalue"] = sig["pipeline-transcriptome-de_p-value"]
    sig["adj_pvalue"] = sig["pipeline-transcriptome-de_adjusted p-value"]
    sig["minerals"] = sig["gene_symbol"].map(gene_to_minerals).fillna("")
    rna_sig_all.append(sig[["gene_symbol", "omics", "comparison", "log2FC", "pvalue", "adj_pvalue", "minerals"]])
    print(f"  {sheet}: {len(sig)} significant mineral-pathway genes")

rna_sig = pd.concat(rna_sig_all, ignore_index=True)

# ── Proteomics ──
print("\n=== Proteomics mineral-pathway DE ===")
prot_de = pd.read_csv(f"{PROC}/astronaut_proteomics_de.csv")
print(f"  Columns: {list(prot_de.columns)}")
print(f"  Sheets: {prot_de['_sheet'].unique() if '_sheet' in prot_de.columns else 'N/A'}")
prot_de.head()


=== RNA-seq mineral-pathway DE (pipeline-transcriptome-de) ===


  I4-FP1: 20 significant mineral-pathway genes
  I4-FP2: 10 significant mineral-pathway genes
  I4-FP3: 16 significant mineral-pathway genes

=== Proteomics mineral-pathway DE ===


  Columns: ['Gene', 'logFC', 'AveExpr', 't', 'P.Value', 'adj.P.Val', 'B', '_sheet', '_comparison']
  Sheets: ['I4-FP1' 'I4-FP2']


,Gene,logFC,AveExpr,t,P.Value,adj.P.Val,B,_sheet,_comparison
0,PF4,1.445671,18.630286,7.957981,2.589947e-07,0.000457,7.043966,I4-FP1,"(R+1) vs (L-92, L-44, L-3)"
1,QPCT,-1.604759,10.699338,-6.981444,1.584531e-06,0.001045,5.373022,I4-FP1,"(R+1) vs (L-92, L-44, L-3)"
2,TGFB1,1.483192,16.085805,6.922152,1.776072e-06,0.001045,5.266779,I4-FP1,"(R+1) vs (L-92, L-44, L-3)"
3,APP,1.320346,12.910695,6.725544,2.601841e-06,0.001148,4.910578,I4-FP1,"(R+1) vs (L-92, L-44, L-3)"
4,CCN2,1.277208,13.223742,6.366990,5.290941e-06,0.001868,4.245643,I4-FP1,"(R+1) vs (L-92, L-44, L-3)"


In [18]:
# ── Proteomics mineral-pathway cross-reference ──
prot_de = pd.read_csv(f"{PROC}/astronaut_proteomics_de.csv")
mineral_genes_set = set(crosswalk["gene_symbol"].dropna())
prot_de["is_mineral_gene"] = prot_de["Gene"].isin(mineral_genes_set)
prot_mineral = prot_de[prot_de["is_mineral_gene"]].copy()
print(f"Proteomics mineral-pathway proteins: {len(prot_mineral)}")

prot_sig_all = []
for sheet in prot_mineral["_sheet"].unique():
    sub = prot_mineral[prot_mineral["_sheet"] == sheet].copy()
    sig = sub[(sub["logFC"].abs() > 0.5) & (sub["P.Value"] < 0.05)].copy()
    sig["omics"] = "Proteomics"
    sig["comparison"] = sheet
    sig["gene_symbol"] = sig["Gene"]
    sig["log2FC"] = sig["logFC"]
    sig["pvalue"] = sig["P.Value"]
    sig["adj_pvalue"] = sig["adj.P.Val"]
    sig["minerals"] = sig["gene_symbol"].map(gene_to_minerals).fillna("")
    prot_sig_all.append(sig[["gene_symbol", "omics", "comparison", "log2FC", "pvalue", "adj_pvalue", "minerals"]])
    print(f"  {sheet}: {len(sig)} significant mineral-pathway proteins")
    if len(sig) > 0:
        print(sig[["Gene", "logFC", "P.Value", "adj.P.Val", "minerals"]].head(10).to_string(index=False))

prot_sig = pd.concat(prot_sig_all, ignore_index=True) if prot_sig_all else pd.DataFrame()

# ── Metabolomics (no gene symbols, but check for mineral-related metabolites) ──
print("\n=== Metabolomics DE ===")
metab_de = pd.read_csv(f"{PROC}/astronaut_metabolomics_de.csv")
print(f"  Columns: {list(metab_de.columns)}")
print(f"  Rows: {len(metab_de)}")
print(metab_de.head(3).to_string(index=False))

# ── Combine RNA + Proteomics mineral-pathway DE ──
all_sig = pd.concat([rna_sig, prot_sig], ignore_index=True)
all_sig = all_sig.sort_values("pvalue")
print(f"\n=== Combined mineral-pathway DE: {len(all_sig)} significant hits ===")
print(f"  RNA-seq: {len(rna_sig)}, Proteomics: {len(prot_sig)}")
print(f"  Unique genes: {all_sig['gene_symbol'].nunique()}")
print(f"  Minerals represented: {sorted(set(m for s in all_sig['minerals'].dropna() for m in s.split(';'))) if all_sig['minerals'].notna().any() else 'N/A'}")

# Save combined DE
all_sig.to_csv(f"{PROC}/mineral_pathway_de_combined.csv", index=False)
print(f"  Saved to mineral_pathway_de_combined.csv")

# Show top 20
print("\nTop 20 mineral-pathway DE hits:")
print(all_sig.head(20).to_string(index=False))


Proteomics mineral-pathway proteins: 122
  I4-FP1: 12 significant mineral-pathway proteins
   Gene     logFC  P.Value  adj.P.Val  minerals
    APP  1.320346 0.000003   0.001148    Copper
  PDGFA  1.059941 0.000066   0.006423          
  FXYD2 -0.930624 0.000583   0.023390    Sodium
   CYBB -0.799565 0.001888   0.052059          
    LOX -0.720257 0.002918   0.069590    Copper
   GCLM  1.047790 0.005272   0.102251          
ATP6AP1 -0.930772 0.013243   0.148783          
 S100A8 -0.531235 0.015984   0.164730 Magnesium
S100A12 -0.997097 0.016155   0.164818 Magnesium
 S100A9 -0.518937 0.016837   0.169816 Magnesium
  I4-FP2: 13 significant mineral-pathway proteins
    Gene     logFC  P.Value  adj.P.Val  minerals
   PDGFA  1.002600 0.000013   0.002560          
     APP  1.090288 0.000030   0.003752    Copper
   FXYD2 -0.892169 0.000250   0.012677    Sodium
     LOX -0.673637 0.000868   0.030673    Copper
 S100A12 -1.196476 0.001953   0.052086 Magnesium
    GCLM  0.959731 0.002090   0.05430

  Columns: ['ID', 'logFC', 'AveExpr', 't', 'P.Value', 'adj.P.Val', 'B', 'col_7', '_sheet', '_comparison', 'col_8']
  Rows: 2624
          ID     logFC      AveExpr         t      P.Value  adj.P.Val        B  col_7 _sheet                _comparison  col_8
LysoPC(15:0) -2.341561 2.711373e-15 -6.721362 5.614970e-08   0.000037 8.306020    0.0 I4-FP1 (R+1) vs (L-92, L-44, L-3)    NaN
LysoPC(14:0) -2.108057 2.225650e-15 -5.938794 6.681835e-07   0.000217 5.943995    0.0 I4-FP1 (R+1) vs (L-92, L-44, L-3)    NaN
LysoPC(17:0) -2.046564 1.492151e-15 -5.813748 9.936616e-07   0.000217 5.565445    0.0 I4-FP1 (R+1) vs (L-92, L-44, L-3)    NaN

=== Combined mineral-pathway DE: 71 significant hits ===
  RNA-seq: 46, Proteomics: 25
  Unique genes: 50
  Minerals represented: ['', 'Calcium', 'Copper', 'Iron', 'Magnesium', 'Potassium', 'Selenium', 'Sodium', 'Zinc']


  Saved to mineral_pathway_de_combined.csv

Top 20 mineral-pathway DE hits:
gene_symbol      omics comparison    log2FC   pvalue  adj_pvalue  minerals
        APP Proteomics     I4-FP1  1.320346 0.000003    0.001148    Copper
      ALAS2    RNA-seq     I4-FP1 -1.446597 0.000010    0.008249      Iron
      PDGFA Proteomics     I4-FP2  1.002600 0.000013    0.002560          
        APP Proteomics     I4-FP2  1.090288 0.000030    0.003752    Copper
      PDGFA Proteomics     I4-FP1  1.059941 0.000066    0.006423          
      FXYD2 Proteomics     I4-FP2 -0.892169 0.000250    0.012677    Sodium
      GLRX5    RNA-seq     I4-FP1 -1.057519 0.000302    0.083373      Iron
      FXYD2 Proteomics     I4-FP1 -0.930624 0.000583    0.023390    Sodium
        LOX Proteomics     I4-FP2 -0.673637 0.000868    0.030673    Copper
       CYBB Proteomics     I4-FP1 -0.799565 0.001888    0.052059          
    S100A12 Proteomics     I4-FP2 -1.196476 0.001953    0.052086 Magnesium
       GCLM Proteomics  

In [19]:
# ── Stage 3: TabPFN ML ──
# Build feature matrices from RNA-seq counts + mineral pathway genes
import pandas as pd, numpy as np

PROC = "/mnt/results/data"

# Load count data
print("Loading RNA-seq counts...")
rna_counts = pd.read_csv(f"{PROC}/astronaut_rnaseq_counts.csv", index_col=0)
print(f"  Shape: {rna_counts.shape}")
print(f"  Columns: {list(rna_counts.columns[:10])}...")
print(f"  Index sample: {list(rna_counts.index[:3])}")

# Load ENSEMBL mapping
ens_map = pd.read_csv(f"{PROC}/ensembl_to_symbol.csv")
ens_to_sym = dict(zip(ens_map["ensembl_id"], ens_map["gene_symbol"]))

# Load mineral genes
crosswalk = pd.read_csv(f"{PROC}/mineral_pathway_crosswalk.csv")
mineral_genes = set(crosswalk["gene_symbol"].dropna())

# Map counts index to symbols
counts_sym = rna_counts.copy()
counts_sym["gene_symbol"] = counts_sym.index.map(lambda x: ens_to_sym.get(x, ""))
counts_sym = counts_sym[counts_sym["gene_symbol"] != ""].set_index("gene_symbol")
print(f"  After symbol mapping: {counts_sym.shape}")

# Filter to mineral-pathway genes only
mineral_counts = counts_sym[counts_sym.index.isin(mineral_genes)]
print(f"  Mineral-pathway genes in counts: {mineral_counts.shape}")

# Parse column names to get subject + timepoint + flight/ground label
# Columns are like C001_L-92, C001_R+1, etc.
cols_info = []
for col in mineral_counts.columns:
    parts = col.split("_")
    if len(parts) >= 2:
        subject = parts[0]
        timepoint = "_".join(parts[1:])
        is_flight = timepoint.startswith("R+")
        cols_info.append({"col": col, "subject": subject, "timepoint": timepoint, "is_flight": is_flight})
    else:
        cols_info.append({"col": col, "subject": "", "timepoint": "", "is_flight": False})

cols_df = pd.DataFrame(cols_info)
print(f"\n  Sample columns info:")
print(cols_df.head(10).to_string(index=False))
print(f"  Flight samples: {cols_df['is_flight'].sum()}, Ground samples: {(~cols_df['is_flight']).sum()}")


Loading RNA-seq counts...


  Shape: (61852, 16)
  Columns: ['C001_L-92', 'C001_L-44', 'C001_L-3', 'C001_R+1', 'C002_L-92', 'C002_L-44', 'C002_L-3', 'C002_R+1', 'C003_L-92', 'C003_L-44']...
  Index sample: ['ENSG00000000003.15', 'ENSG00000000005.6', 'ENSG00000000419.14']


  After symbol mapping: (61852, 16)
  Mineral-pathway genes in counts: (786, 16)

  Sample columns info:
      col subject timepoint  is_flight
C001_L-92    C001      L-92      False
C001_L-44    C001      L-44      False
 C001_L-3    C001       L-3      False
 C001_R+1    C001       R+1       True
C002_L-92    C002      L-92      False
C002_L-44    C002      L-44      False
 C002_L-3    C002       L-3      False
 C002_R+1    C002       R+1       True
C003_L-92    C003      L-92      False
C003_L-44    C003      L-44      False
  Flight samples: 4, Ground samples: 12


In [20]:
# ── Stage 3: ML Analysis ──
# Primary: sklearn classifiers (RandomForest, SVM, LogisticRegression)
# TabPFN was intended but model weights couldn't be downloaded in sandbox
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.svm import SVC, SVR
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

PROC = "/mnt/results/data"

# Aggregate duplicate gene symbols
mineral_counts_agg = mineral_counts.groupby(mineral_counts.index).sum()
print(f"After aggregating duplicates: {mineral_counts_agg.shape}")

# Prepare feature matrix (samples × genes)
X = mineral_counts_agg.T
y = np.array([1 if c else 0 for c in cols_df["is_flight"]])

# Remove zero-variance genes, select top 200 by variance
var = X.var(axis=0)
X_var = X.loc[:, var > 0]
top_var = X_var.var(axis=0).nlargest(200).index
X_sel = X_var[top_var]
print(f"Feature matrix: {X_sel.shape} (samples × genes)")
print(f"  Flight: {y.sum()}, Ground: {(~y.astype(bool)).sum()}")

# Standardize
scaler = StandardScaler()
X_arr = scaler.fit_transform(X_sel.values)
feature_names = list(X_sel.columns)

# ── Classification: Flight vs Ground (LOO-CV) ──
print("\n=== Classification: Flight vs Ground (LOO-CV) ===")
classifiers = {
    "RandomForest": RandomForestClassifier(n_estimators=500, random_state=42, max_depth=5),
    "SVM_RBF": SVC(kernel='rbf', probability=True, random_state=42, C=1.0),
    "SVM_Linear": SVC(kernel='linear', probability=True, random_state=42, C=1.0),
    "LogisticRegression": LogisticRegression(max_iter=5000, random_state=42, C=1.0),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=3),
}

loo = LeaveOneOut()
all_results = {}
for name, clf in classifiers.items():
    y_pred = []
    y_proba = []
    for train_idx, test_idx in loo.split(X_arr):
        clf.fit(X_arr[train_idx], y[train_idx])
        y_pred.append(clf.predict(X_arr[test_idx])[0])
        if hasattr(clf, 'predict_proba'):
            y_proba.append(clf.predict_proba(X_arr[test_idx])[0, 1])
        else:
            y_proba.append(clf.decision_function(X_arr[test_idx])[0])
    y_pred = np.array(y_pred)
    y_proba = np.array(y_proba)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y, y_proba)
    except:
        auc = np.nan
    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    all_results[name] = {"acc": acc, "f1": f1, "auc": auc, "pred": y_pred, "proba": y_proba, "cm": cm}
    print(f"  {name}: Acc={acc:.3f}, F1={f1:.3f}, AUC={auc:.3f}")
    print(f"    CM: {cm.tolist()} (TN/FP, FN/TP)")

# Save best model results
best_name = max(all_results, key=lambda k: all_results[k]['auc'] if not np.isnan(all_results[k]['auc']) else all_results[k]['acc'])
best = all_results[best_name]
print(f"\nBest model: {best_name} (AUC={best['auc']:.3f})")

classification_results = pd.DataFrame({
    "sample": list(X_sel.index),
    "true_label": y,
    "predicted_label": best["pred"],
    "flight_probability": best["proba"],
})
classification_results.to_csv(f"{PROC}/ml_classification_results.csv", index=False)

model_comparison = pd.DataFrame([
    {"model": name, "accuracy": r["acc"], "f1": r["f1"], "auc": r["auc"]}
    for name, r in all_results.items()
])
model_comparison.to_csv(f"{PROC}/ml_model_comparison.csv", index=False)
print(f"  Saved classification results and model comparison")


After aggregating duplicates: (784, 16)
Feature matrix: (16, 200) (samples × genes)
  Flight: 4, Ground: 12

=== Classification: Flight vs Ground (LOO-CV) ===


  RandomForest: Acc=0.750, F1=0.000, AUC=0.208
    CM: [[12, 0], [4, 0]] (TN/FP, FN/TP)
  SVM_RBF: Acc=0.750, F1=0.000, AUC=0.312
    CM: [[12, 0], [4, 0]] (TN/FP, FN/TP)
  SVM_Linear: Acc=0.688, F1=0.000, AUC=0.000
    CM: [[11, 1], [4, 0]] (TN/FP, FN/TP)
  LogisticRegression: Acc=0.750, F1=0.000, AUC=0.250
    CM: [[12, 0], [4, 0]] (TN/FP, FN/TP)


  GradientBoosting: Acc=0.188, F1=0.000, AUC=0.000
    CM: [[3, 9], [4, 0]] (TN/FP, FN/TP)

Best model: SVM_RBF (AUC=0.312)


  Saved classification results and model comparison


In [21]:
# ── Classification with balanced weights + Regression of mineral levels ──
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix, r2_score, mean_squared_error
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

PROC = "/mnt/results/data"

# ── Classification with balanced class weights ──
print("=== Classification: Flight vs Ground (balanced, LOO-CV) ===")
classifiers_bal = {
    "RandomForest_bal": RandomForestClassifier(n_estimators=500, random_state=42, max_depth=5, class_weight='balanced'),
    "SVM_RBF_bal": SVC(kernel='rbf', probability=True, random_state=42, C=1.0, class_weight='balanced'),
    "SVM_Linear_bal": SVC(kernel='linear', probability=True, random_state=42, C=1.0, class_weight='balanced'),
    "LogReg_bal": LogisticRegression(max_iter=5000, random_state=42, C=1.0, class_weight='balanced'),
}

loo = LeaveOneOut()
bal_results = {}
for name, clf in classifiers_bal.items():
    y_pred = []
    y_proba = []
    for train_idx, test_idx in loo.split(X_arr):
        clf.fit(X_arr[train_idx], y[train_idx])
        y_pred.append(clf.predict(X_arr[test_idx])[0])
        y_proba.append(clf.predict_proba(X_arr[test_idx])[0, 1])
    y_pred = np.array(y_pred)
    y_proba = np.array(y_proba)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, zero_division=0)
    auc = roc_auc_score(y, y_proba)
    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    bal_results[name] = {"acc": acc, "f1": f1, "auc": auc, "pred": y_pred, "proba": y_proba, "cm": cm}
    print(f"  {name}: Acc={acc:.3f}, F1={f1:.3f}, AUC={auc:.3f}")
    print(f"    CM: {cm.tolist()} (TN/FP, FN/TP)")

# ── Regression: Predict mineral levels from gene expression ──
print("\n=== Regression: Mineral levels from gene expression ===")
# Load CMP mineral data
mineral_combined = pd.read_csv(f"{PROC}/astronaut_minerals_combined.csv")
print(f"Mineral data columns: {list(mineral_combined.columns)}")
print(mineral_combined.head())

# Load iron proxy
iron_proxy = pd.read_csv(f"{PROC}/astronaut_cbc_iron_proxy.csv")
print(f"\nIron proxy columns: {list(iron_proxy.columns)}")
print(iron_proxy.head())


=== Classification: Flight vs Ground (balanced, LOO-CV) ===


  RandomForest_bal: Acc=0.562, F1=0.000, AUC=0.000
    CM: [[9, 3], [4, 0]] (TN/FP, FN/TP)
  SVM_RBF_bal: Acc=0.750, F1=0.000, AUC=0.604
    CM: [[12, 0], [4, 0]] (TN/FP, FN/TP)
  SVM_Linear_bal: Acc=0.688, F1=0.000, AUC=0.000
    CM: [[11, 1], [4, 0]] (TN/FP, FN/TP)
  LogReg_bal: Acc=0.625, F1=0.000, AUC=0.292
    CM: [[10, 2], [4, 0]] (TN/FP, FN/TP)

=== Regression: Mineral levels from gene expression ===


Mineral data columns: ['SUBJECT_ID', 'timepoint', 'CALCIUM', 'POTASSIUM', 'SODIUM', 'CHLORIDE', 'CARBON DIOXIDE', 'HEMOGLOBIN', 'HEMATOCRIT', 'RED BLOOD CELL COUNT', 'MCH', 'MCHC', 'MCV']
  SUBJECT_ID timepoint  CALCIUM  POTASSIUM  SODIUM  CHLORIDE  CARBON DIOXIDE  \
0       C001       L-3      9.6        3.9   142.0     102.0            26.0   
1       C001      L-44     10.0        4.2   139.0     101.0            26.0   
2       C001      L-92      9.7        4.6   141.0     104.0            23.0   
3       C001       R+1      9.7        4.4   138.0     102.0            24.0   
4       C001     R+194      9.4        4.3   137.0     102.0            23.0   

   HEMOGLOBIN  HEMATOCRIT  RED BLOOD CELL COUNT   MCH  MCHC   MCV  
0        14.7        44.5                  4.69  31.3  33.0  94.9  
1        14.9        45.1                  4.85  30.7  33.0  93.0  
2        14.1        42.5                  4.58  30.8  33.2  92.8  
3        14.7        43.6                  4.84  30.4  33.7

In [22]:
# ── Regression: Predict mineral levels from gene expression ──
print("=== Regression: Mineral levels from gene expression ===")

# Build matched dataset
matched_data = []
for col in X_sel.index:
    parts = col.split("_")
    subject = parts[0]
    timepoint = "_".join(parts[1:])
    mineral_row = mineral_combined[(mineral_combined["SUBJECT_ID"] == subject) & 
                                    (mineral_combined["timepoint"] == timepoint)]
    if len(mineral_row) > 0:
        row = {"sample": col, "subject": subject, "timepoint": timepoint}
        row.update(mineral_row.iloc[0][["CALCIUM", "POTASSIUM", "SODIUM", "CHLORIDE", "CARBON DIOXIDE", "HEMOGLOBIN"]].to_dict())
        matched_data.append(row)

matched_df = pd.DataFrame(matched_data)
print(f"Matched samples: {len(matched_df)}")

# Regression targets (handle NaN by imputing with subject mean)
reg_targets = ["CALCIUM", "POTASSIUM", "SODIUM", "HEMOGLOBIN"]
for target in reg_targets:
    vals = matched_df[target].values.astype(float)
    if np.any(np.isnan(vals)):
        # Impute with subject mean
        for subj in matched_df["subject"].unique():
            mask = (matched_df["subject"] == subj) & matched_df[target].notna()
            subj_mean = matched_df.loc[mask, target].mean()
            nan_mask = (matched_df["subject"] == subj) & matched_df[target].isna()
            matched_df.loc[nan_mask, target] = subj_mean
        print(f"  Imputed NaN in {target}")

print(matched_df[["sample", "CALCIUM", "POTASSIUM", "SODIUM", "HEMOGLOBIN"]].to_string(index=False))

reg_results = []
for target in reg_targets:
    y_reg = matched_df[target].values.astype(float)
    for reg_name, reg in [
        ("Ridge", Ridge(alpha=1.0)),
        ("SVR_linear", SVR(kernel='linear', C=1.0)),
        ("SVR_rbf", SVR(kernel='rbf', C=1.0)),
        ("RF", RandomForestRegressor(n_estimators=200, random_state=42, max_depth=5)),
    ]:
        y_pred_reg = []
        for train_idx, test_idx in loo.split(X_arr):
            reg.fit(X_arr[train_idx], y_reg[train_idx])
            y_pred_reg.append(reg.predict(X_arr[test_idx])[0])
        y_pred_reg = np.array(y_pred_reg)
        r2 = r2_score(y_reg, y_pred_reg)
        rmse = np.sqrt(mean_squared_error(y_reg, y_pred_reg))
        corr = np.corrcoef(y_reg, y_pred_reg)[0, 1] if np.std(y_pred_reg) > 0 else 0
        reg_results.append({
            "target": target, "model": reg_name, "r2": r2, "rmse": rmse, "correlation": corr,
            "y_true": y_reg.tolist(), "y_pred": y_pred_reg.tolist(),
        })
        print(f"  {target} / {reg_name}: R²={r2:.3f}, RMSE={rmse:.3f}, r={corr:.3f}")

# Save
reg_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ["y_true", "y_pred"]} for r in reg_results])
reg_df.to_csv(f"{PROC}/ml_regression_results.csv", index=False)

print("\nBest model per target:")
for target in reg_targets:
    target_results = [r for r in reg_results if r["target"] == target]
    best_r = max(target_results, key=lambda x: x["correlation"])
    print(f"  {target}: {best_r['model']} (r={best_r['correlation']:.3f}, R²={best_r['r2']:.3f})")


=== Regression: Mineral levels from gene expression ===
Matched samples: 16
  Imputed NaN in HEMOGLOBIN
   sample  CALCIUM  POTASSIUM  SODIUM  HEMOGLOBIN
C001_L-92      9.7        4.6   141.0   14.100000
C001_L-44     10.0        4.2   139.0   14.900000
 C001_L-3      9.6        3.9   142.0   14.700000
 C001_R+1      9.7        4.4   138.0   14.700000
C002_L-92     10.2        3.9   142.0   13.200000
C002_L-44      9.7        4.0   141.0   12.800000
 C002_L-3      9.3        3.8   143.0   11.500000
 C002_R+1      9.2        3.5   137.0   12.100000
C003_L-92      9.3        3.7   139.0   13.200000
C003_L-44     10.3        3.5   139.0   14.400000
 C003_L-3      9.5        3.3   138.0   13.300000
 C003_R+1      9.4        3.0   137.0   13.633333
C004_L-92     10.0        4.2   138.0   17.000000
C004_L-44     10.1        4.2   140.0   16.600000
 C004_L-3     10.2        3.7   141.0   16.800000
 C004_R+1      9.8        4.0   140.0   16.500000
  CALCIUM / Ridge: R²=-0.592, RMSE=0.433, r=-0

  CALCIUM / RF: R²=-0.330, RMSE=0.395, r=-0.255
  POTASSIUM / Ridge: R²=0.122, RMSE=0.377, r=0.500
  POTASSIUM / SVR_linear: R²=0.115, RMSE=0.379, r=0.455
  POTASSIUM / SVR_rbf: R²=0.163, RMSE=0.368, r=0.412


  POTASSIUM / RF: R²=-0.224, RMSE=0.446, r=-0.361
  SODIUM / Ridge: R²=0.159, RMSE=1.644, r=0.432
  SODIUM / SVR_linear: R²=0.193, RMSE=1.611, r=0.460
  SODIUM / SVR_rbf: R²=-0.388, RMSE=2.112, r=-0.571


  SODIUM / RF: R²=-0.586, RMSE=2.258, r=-0.514
  HEMOGLOBIN / Ridge: R²=-0.594, RMSE=2.074, r=0.475
  HEMOGLOBIN / SVR_linear: R²=-0.615, RMSE=2.087, r=0.477
  HEMOGLOBIN / SVR_rbf: R²=-0.093, RMSE=1.717, r=-0.063


  HEMOGLOBIN / RF: R²=-0.381, RMSE=1.930, r=-0.138



Best model per target:
  CALCIUM: Ridge (r=-0.015, R²=-0.592)
  POTASSIUM: Ridge (r=0.500, R²=0.122)
  SODIUM: SVR_linear (r=0.460, R²=0.193)
  HEMOGLOBIN: SVR_linear (r=0.477, R²=-0.615)


In [23]:
# ── Permutation Importance: identify key driver genes ──
print("=== Permutation Importance ===")

# For classification (SVM_RBF_bal was best with AUC=0.604)
from sklearn.svm import SVC
clf_imp = SVC(kernel='rbf', probability=True, random_state=42, C=1.0, class_weight='balanced')
clf_imp.fit(X_arr, y)
result_clf = permutation_importance(clf_imp, X_arr, y, n_repeats=20, random_state=42, scoring='roc_auc')

clf_imp_df = pd.DataFrame({
    "gene": feature_names,
    "importance_mean": result_clf.importances_mean,
    "importance_std": result_clf.importances_std,
}).sort_values("importance_mean", ascending=False)

# Add mineral annotations
gene_to_minerals_map = dict(zip(crosswalk["gene_symbol"], crosswalk["minerals"]))
clf_imp_df["minerals"] = clf_imp_df["gene"].map(gene_to_minerals_map).fillna("")

print("\nTop 20 genes for flight/ground classification:")
print(clf_imp_df.head(20).to_string(index=False))
clf_imp_df.to_csv(f"{PROC}/permutation_importance_classification.csv", index=False)

# For regression (POTASSIUM - best target)
print("\n=== Permutation Importance for POTASSIUM regression ===")
y_k = matched_df["POTASSIUM"].values.astype(float)
reg_imp = Ridge(alpha=1.0)
reg_imp.fit(X_arr, y_k)
result_reg = permutation_importance(reg_imp, X_arr, y_k, n_repeats=20, random_state=42, scoring='r2')

reg_imp_df = pd.DataFrame({
    "gene": feature_names,
    "importance_mean": result_reg.importances_mean,
    "importance_std": result_reg.importances_std,
}).sort_values("importance_mean", ascending=False)
reg_imp_df["minerals"] = reg_imp_df["gene"].map(gene_to_minerals_map).fillna("")

print("Top 20 genes for POTASSIUM regression:")
print(reg_imp_df.head(20).to_string(index=False))
reg_imp_df.to_csv(f"{PROC}/permutation_importance_potassium.csv", index=False)

# For SODIUM regression
print("\n=== Permutation Importance for SODIUM regression ===")
y_na = matched_df["SODIUM"].values.astype(float)
reg_imp_na = Ridge(alpha=1.0)
reg_imp_na.fit(X_arr, y_na)
result_na = permutation_importance(reg_imp_na, X_arr, y_na, n_repeats=20, random_state=42, scoring='r2')

na_imp_df = pd.DataFrame({
    "gene": feature_names,
    "importance_mean": result_na.importances_mean,
    "importance_std": result_na.importances_std,
}).sort_values("importance_mean", ascending=False)
na_imp_df["minerals"] = na_imp_df["gene"].map(gene_to_minerals_map).fillna("")

print("Top 20 genes for SODIUM regression:")
print(na_imp_df.head(20).to_string(index=False))
na_imp_df.to_csv(f"{PROC}/permutation_importance_sodium.csv", index=False)

print("\nSaved all permutation importance results")


=== Permutation Importance ===



Top 20 genes for flight/ground classification:
    gene  importance_mean  importance_std  minerals
   ALAS2         0.002083         0.00625      Iron
  S100A9         0.000000         0.00000 Magnesium
    FTH1         0.000000         0.00000      Iron
     FTL         0.000000         0.00000      Iron
SLC25A37         0.000000         0.00000      Iron
 S100A11         0.000000         0.00000 Magnesium
  S100A6         0.000000         0.00000 Magnesium
  S100A8         0.000000         0.00000 Magnesium
  S100A4         0.000000         0.00000 Magnesium
    GPX1         0.000000         0.00000  Selenium
    GNAS         0.000000         0.00000          
   PCBP1         0.000000         0.00000          
 SLC25A6         0.000000         0.00000          
   MSRB1         0.000000         0.00000  Selenium
   NCOA4         0.000000         0.00000      Iron
 ATP5F1E         0.000000         0.00000          
   PCBP2         0.000000         0.00000          
    CYBB        


=== Permutation Importance for POTASSIUM regression ===


Top 20 genes for POTASSIUM regression:
    gene  importance_mean  importance_std           minerals
 SLC30A1         0.058145        0.016167               Zinc
   UQCRB         0.043350        0.011766                   
  NDUFB7         0.042997        0.012703                   
 ATP5F1D         0.034754        0.009897                   
   CAMK1         0.030744        0.010120                   
  SLC4A7         0.030390        0.006376             Sodium
   VEGFB         0.029123        0.006702                   
  ATP5PD         0.027550        0.008981                   
 NDUFB11         0.026960        0.008152                   
   CXCR4         0.023479        0.007500                   
 SLC12A9         0.021651        0.007247             Sodium
  NDUFV1         0.021537        0.006546                   
    ALPL         0.020897        0.005553 Calcium;Phosphorus
   COX6C         0.020272        0.006196                   
ATP6V0E2         0.019968        0.004811     


=== Permutation Importance for SODIUM regression ===


Top 20 genes for SODIUM regression:
   gene  importance_mean  importance_std           minerals
  GLRX5         0.079741        0.022720               Iron
 NFATC3         0.067401        0.018582                   
SELENON         0.055552        0.015856           Selenium
 NDUFV1         0.054086        0.015616                   
   GPX1         0.048270        0.015098           Selenium
  CAMK4         0.039477        0.009453            Calcium
  ITPKB         0.038750        0.010065                   
 SLC2A1         0.034103        0.007144                   
 PPP3R1         0.031816        0.010203            Calcium
ATP6V0B         0.029651        0.009285                   
SLC30A7         0.028563        0.006657               Zinc
  UQCRH         0.028433        0.007772                   
SELENOW         0.027356        0.007013           Selenium
SLC11A1         0.026300        0.007833               Iron
  PCBP2         0.025665        0.005858                   
SLC3


Saved all permutation importance results


In [24]:
from nhanes_data.nhanes_data_api import NHANESDataAPI
nhanes_api = NHANESDataAPI()

# List available data categories
categories = nhanes_api.list_data_categories()
print('Data categories:', categories)

# List available cycles
cycles = nhanes_api.list_cycle_years()
print('Cycles:', cycles)

Data categories: ['demographics', 'dietary', 'examination', 'laboratory', 'questionnaire', 'limitedaccess']
Cycles: ['1999-2000', '2001-2002', '2003-2004', '2005-2006', '2007-2008', '2009-2010', '2011-2012', '2013-2014', '2015-2016', '2017-2018']


In [25]:
# List laboratory files for the latest cycle (2017-2018)
lab_files_2018 = nhanes_api.list_file_names('laboratory', '2017-2018')
print('Lab files 2017-2018:')
for f in lab_files_2018:
    print(f'  {f}')

Lab files 2017-2018:
  Urine Flow Rate
  Fasting Questionnaire
  Glycohemoglobin
  Albumin & Creatinine - Urine
  Urine Pregnancy Test
  HIV Antibody Test
  Hepatitis A
  Hepatitis B: Surface Antibody
  Hepatitis B: Core antibody, Surface antigen, and Hepatitis D: antibody
  Hepatitis C: RNA (HCV-RNA), Confirmed Antibody (INNO-LIA), & Genotype
  Hepatitis E: IgG & IgM Antibodies
  Complete Blood Count with 5-Part Differential
  High-Sensitivity C-Reactive Protein
  Cholesterol - Total
  Cholesterol - High - Density Lipoprotein (HDL)
  Standard Biochemistry Profile
  Iron Status - Serum
  Lead, Cadmium, Total Mercury, Selenium, & Manganese - Blood
  Chromium & Cobalt
  Mercury: Inorganic, Ethyl and Methyl - Blood
  Cytomegalovirus IgG & IgM Antibodies - Serum
  Herpes Simplex Virus Type-2 - Youth
  Herpes Simplex Virus Type-1 and Type-2
  Mycoplasma genitalium - Urine - Adult
  Mycoplasma genitalium - Urine - Youth
  Trichomonas - Urine - Adult
  Trichomonas - Urine - Youth
  Chlamydia 

In [26]:
# Retrieve the Standard Biochemistry Profile (CMP) for 2017-2018
cmp = nhanes_api.retrieve_data('laboratory', '2017-2018', 'Standard Biochemistry Profile')
print('CMP shape:', cmp.shape)
print('CMP columns:')
for c in cmp.columns:
    print(f'  {c}')
print()
print(cmp.head(3))

ValueError: Header record is not an XPORT file.

In [27]:
import urllib.request, os, pandas as pd
from io import BytesIO

# NHANES 2017-2018 XPT file URLs from CDC
# Standard Biochemistry Profile: BIOPRO_J
# Complete Blood Count: CBC_J
# Demographics: DEMO_J
base_url = 'https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/'

files = {
    'BIOPRO_J': base_url + 'BIOPRO_J.XPT',   # Standard Biochemistry Profile
    'CBC_J':    base_url + 'CBC_J.XPT',        # Complete Blood Count
    'DEMO_J':   base_url + 'DEMO_J.XPT',       # Demographics
}

os.makedirs('/workspace/nhanes_raw', exist_ok=True)

downloaded = {}
for name, url in files.items():
    outpath = f'/workspace/nhanes_raw/{name}.XPT'
    if not os.path.exists(outpath):
        print(f'Downloading {name}...')
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            with open(outpath, 'wb') as f:
                f.write(data)
            print(f'  Downloaded {len(data)} bytes')
        except Exception as e:
            print(f'  FAILED: {e}')
            continue
    else:
        print(f'{name} already exists ({os.path.getsize(outpath)} bytes)')
    downloaded[name] = outpath

print('Done downloading.')

  Downloaded 20905 bytes


  Downloaded 20905 bytes


  Downloaded 20905 bytes
Done downloading.


In [28]:
import urllib.request, os

# Try different URL patterns for NHANES XPT files
urls_to_try = {
    'BIOPRO_J': [
        'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BIOPRO_J.XPT',
        'https://wwwn.cdc.gov/nchs/data/nhanes/public/2017/datafiles/BIOPRO_J.XPT',
        'https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/BIOPRO_J.XPT',
    ],
    'CBC_J': [
        'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/CBC_J.XPT',
        'https://wwwn.cdc.gov/nchs/data/nhanes/public/2017/datafiles/CBC_J.XPT',
        'https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/CBC_J.XPT',
    ],
    'DEMO_J': [
        'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.XPT',
        'https://wwwn.cdc.gov/nchs/data/nhanes/public/2017/datafiles/DEMO_J.XPT',
        'https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/DEMO_J.XPT',
    ],
}

os.makedirs('/workspace/nhanes_raw', exist_ok=True)

for name, urls in urls_to_try.items():
    outpath = f'/workspace/nhanes_raw/{name}.XPT'
    success = False
    for url in urls:
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            # Check if it's actually an XPT file (starts with 'HEADER RECORD')
            if data[:12] == b'HEADER RECORD':
                with open(outpath, 'wb') as f:
                    f.write(data)
                print(f'{name}: OK from {url} ({len(data)} bytes)')
                success = True
                break
            else:
                print(f'{name}: {url} returned {len(data)} bytes but NOT XPT (starts with: {data[:50]})')
        except Exception as e:
            print(f'{name}: {url} FAILED: {e}')
    if not success:
        print(f'{name}: ALL URLs FAILED')

BIOPRO_J: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BIOPRO_J.XPT returned 2106080 bytes but NOT XPT (starts with: b'HEADER RECORD*******LIBRARY HEADER RECORD!!!!!!!00')


BIOPRO_J: https://wwwn.cdc.gov/nchs/data/nhanes/public/2017/datafiles/BIOPRO_J.XPT returned 2106080 bytes but NOT XPT (starts with: b'HEADER RECORD*******LIBRARY HEADER RECORD!!!!!!!00')


BIOPRO_J: https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/BIOPRO_J.XPT returned 20905 bytes but NOT XPT (starts with: b'<!DOCTYPE html>\r\n<html lang="en-us" class="cdc-202')
BIOPRO_J: ALL URLs FAILED


CBC_J: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/CBC_J.XPT returned 1476320 bytes but NOT XPT (starts with: b'HEADER RECORD*******LIBRARY HEADER RECORD!!!!!!!00')
CBC_J: https://wwwn.cdc.gov/nchs/data/nhanes/public/2017/datafiles/CBC_J.XPT returned 1476320 bytes but NOT XPT (starts with: b'HEADER RECORD*******LIBRARY HEADER RECORD!!!!!!!00')


CBC_J: https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/CBC_J.XPT returned 20905 bytes but NOT XPT (starts with: b'<!DOCTYPE html>\r\n<html lang="en-us" class="cdc-202')
CBC_J: ALL URLs FAILED


DEMO_J: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.XPT returned 3412720 bytes but NOT XPT (starts with: b'HEADER RECORD*******LIBRARY HEADER RECORD!!!!!!!00')


DEMO_J: https://wwwn.cdc.gov/nchs/data/nhanes/public/2017/datafiles/DEMO_J.XPT returned 3412720 bytes but NOT XPT (starts with: b'HEADER RECORD*******LIBRARY HEADER RECORD!!!!!!!00')
DEMO_J: https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/DEMO_J.XPT returned 20905 bytes but NOT XPT (starts with: b'<!DOCTYPE html>\r\n<html lang="en-us" class="cdc-202')
DEMO_J: ALL URLs FAILED


In [29]:
import urllib.request, os, pandas as pd

# Correct download URLs (first pattern works)
base = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/'
files = {'BIOPRO_J': base + 'BIOPRO_J.XPT', 'CBC_J': base + 'CBC_J.XPT', 'DEMO_J': base + 'DEMO_J.XPT'}

os.makedirs('/workspace/nhanes_raw', exist_ok=True)
for name, url in files.items():
    outpath = f'/workspace/nhanes_raw/{name}.XPT'
    if os.path.getsize(outpath) < 100000:  # re-download if previous attempt was HTML
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=120) as resp:
            data = resp.read()
        with open(outpath, 'wb') as f:
            f.write(data)
        print(f'{name}: downloaded {len(data)} bytes')
    else:
        print(f'{name}: already have {os.path.getsize(outpath)} bytes')

# Parse with pandas
biopro = pd.read_sas('/workspace/nhanes_raw/BIOPRO_J.XPT', format='xport')
cbc = pd.read_sas('/workspace/nhanes_raw/CBC_J.XPT', format='xport')
demo = pd.read_sas('/workspace/nhanes_raw/DEMO_J.XPT', format='xport')

print(f'\nBIOPRO_J: {biopro.shape}')
print(f'CBC_J: {cbc.shape}')
print(f'DEMO_J: {demo.shape}')

BIOPRO_J: downloaded 2106080 bytes
CBC_J: downloaded 1476320 bytes


DEMO_J: downloaded 3412720 bytes

BIOPRO_J: (6401, 41)
CBC_J: (8366, 22)
DEMO_J: (9254, 46)


In [30]:
# Inspect BIOPRO columns (Standard Biochemistry Profile)
print('=== BIOPRO_J columns ===')
for c in biopro.columns:
    print(f'  {c}')

print('\n=== CBC_J columns ===')
for c in cbc.columns:
    print(f'  {c}')

print('\n=== DEMO_J key columns ===')
demo_cols = ['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'RIDRETH3', 'DMDBORN', 'BMXBMI', 'WTMEC2YR', 'SDMVSTRA', 'SDMVPSU']
for c in demo_cols:
    if c in demo.columns:
        print(f'  {c}: {demo[c].describe().to_dict()}')

=== BIOPRO_J columns ===
  SEQN
  LBXSATSI
  LBDSATLC
  LBXSAL
  LBDSALSI
  LBXSAPSI
  LBXSASSI
  LBXSC3SI
  LBXSBU
  LBDSBUSI
  LBXSCLSI
  LBXSCK
  LBXSCR
  LBDSCRSI
  LBXSGB
  LBDSGBSI
  LBXSGL
  LBDSGLSI
  LBXSGTSI
  LBDSGTLC
  LBXSIR
  LBDSIRSI
  LBXSLDSI
  LBXSOSSI
  LBXSPH
  LBDSPHSI
  LBXSKSI
  LBXSNASI
  LBXSTB
  LBDSTBSI
  LBDSTBLC
  LBXSCA
  LBDSCASI
  LBXSCH
  LBDSCHSI
  LBXSTP
  LBDSTPSI
  LBXSTR
  LBDSTRSI
  LBXSUA
  LBDSUASI

=== CBC_J columns ===
  SEQN
  LBXWBCSI
  LBXLYPCT
  LBXMOPCT
  LBXNEPCT
  LBXEOPCT
  LBXBAPCT
  LBDLYMNO
  LBDMONO
  LBDNENO
  LBDEONO
  LBDBANO
  LBXRBCSI
  LBXHGB
  LBXHCT
  LBXMCVSI
  LBXMCHSI
  LBXMC
  LBXRDW
  LBXPLTSI
  LBXMPSI
  LBXNRBC

=== DEMO_J key columns ===
  SEQN: {'count': 9254.0, 'mean': 98329.5, 'std': 2671.5440292085773, 'min': 93703.0, '25%': 96016.25, '50%': 98329.5, '75%': 100642.75, 'max': 102956.0}
  RIAGENDR: {'count': 9254.0, 'mean': 1.5075642965204237, 'std': 0.4999697925862398, 'min': 1.0, '25%': 1.0, '50%': 2.0, '75%': 2

In [31]:
# Complete variable mapping: astronaut → NHANES
var_map = {
    # Biochemistry Profile (BIOPRO_J)
    'CALCIUM':           ('biopro', 'LBXSCA',    'Total Calcium (mg/dL)'),
    'POTASSIUM':         ('biopro', 'LBXSKSI',   'Potassium (mmol/L)'),
    'SODIUM':            ('biopro', 'LBXSNASI',  'Sodium (mmol/L)'),
    'CHLORIDE':          ('biopro', 'LBXSCLSI',  'Chloride (mmol/L)'),
    'CARBON DIOXIDE':    ('biopro', 'LBXSC3SI',  'Bicarbonate (mmol/L)'),
    # Complete Blood Count (CBC_J)
    'HEMOGLOBIN':        ('cbc',    'LBXHGB',    'Hemoglobin (g/dL)'),
    'HEMATOCRIT':        ('cbc',    'LBXHCT',    'Hematocrit (%)'),
    'RED BLOOD CELL COUNT': ('cbc', 'LBXRBCSI',  'RBC count (million cells/uL)'),
    'MCH':               ('cbc',    'LBXMCHSI',  'Mean cell hemoglobin (pg)'),
    'MCHC':              ('cbc',    'LBXMC',     'Mean Cell Hgb Conc. (g/dL)'),
    'MCV':               ('cbc',    'LBXMCVSI',  'Mean cell volume (fL)'),
    'WHITE BLOOD CELL COUNT': ('cbc', 'LBXWBCSI', 'WBC count (1000 cells/uL)'),
    'RDW':               ('cbc',    'LBXRDW',    'Red cell distribution width (%)'),
    'PLATELET COUNT':    ('cbc',    'LBXPLTSI',  'Platelet count (1000 cells/uL)'),
    'MPV':               ('cbc',    'LBXMPSI',   'Mean platelet volume (fL)'),
}

print('Variable mapping (astronaut → NHANES):')
print(f'{"Astronaut var":<28} {"NHANES file":<10} {"NHANES code":<12} {"Description"}')
print('-'*90)
for astro, (src, code, desc) in var_map.items():
    print(f'{astro:<28} {src:<10} {code:<12} {desc}')

# Check NHANES summary stats for these variables
print('\n=== NHANES 2017-2018 summary statistics ===')
for astro, (src, code, desc) in var_map.items():
    df = biopro if src == 'biopro' else cbc
    if code in df.columns:
        vals = df[code].dropna()
        print(f'{astro:<28} n={len(vals):5d}  mean={vals.mean():.2f}  sd={vals.std():.2f}  median={vals.median():.2f}  [P2.5={vals.quantile(0.025):.2f}, P97.5={vals.quantile(0.975):.2f}]')

Variable mapping (astronaut → NHANES):
Astronaut var                NHANES file NHANES code  Description
------------------------------------------------------------------------------------------
CALCIUM                      biopro     LBXSCA       Total Calcium (mg/dL)
POTASSIUM                    biopro     LBXSKSI      Potassium (mmol/L)
SODIUM                       biopro     LBXSNASI     Sodium (mmol/L)
CHLORIDE                     biopro     LBXSCLSI     Chloride (mmol/L)
CARBON DIOXIDE               biopro     LBXSC3SI     Bicarbonate (mmol/L)
HEMOGLOBIN                   cbc        LBXHGB       Hemoglobin (g/dL)
HEMATOCRIT                   cbc        LBXHCT       Hematocrit (%)
RED BLOOD CELL COUNT         cbc        LBXRBCSI     RBC count (million cells/uL)
MCH                          cbc        LBXMCHSI     Mean cell hemoglobin (pg)
MCHC                         cbc        LBXMC        Mean Cell Hgb Conc. (g/dL)
MCV                          cbc        LBXMCVSI     Mean cell 

In [32]:
# Check what BMI column is called in DEMO_J
bmi_cols = [c for c in demo.columns if 'BMI' in c.upper()]
print('BMI-related columns:', bmi_cols)
print('All DEMO columns:', list(demo.columns[:20]))

# BMXBMI is in the examination file, not demographics. Use what we have.
merged = demo[['SEQN', 'RIAGENDR', 'RIDAGEYR']].merge(
    biopro[['SEQN', 'LBXSCA', 'LBXSKSI', 'LBXSNASI', 'LBXSCLSI', 'LBXSC3SI']], on='SEQN', how='inner'
).merge(
    cbc[['SEQN', 'LBXHGB', 'LBXHCT', 'LBXRBCSI', 'LBXMCHSI', 'LBXMC', 'LBXMCVSI', 
         'LBXWBCSI', 'LBXRDW', 'LBXPLTSI', 'LBXMPSI']], on='SEQN', how='inner'
)

print(f'\nMerged dataset: {merged.shape}')
print(f'Age range: {merged["RIDAGEYR"].min():.0f} - {merged["RIDAGEYR"].max():.0f}')
print(f'Sex: {merged["RIAGENDR"].value_counts().to_dict()} (1=Male, 2=Female)')

# Age 40-60 subset (astronaut age range)
age_match = merged[(merged['RIDAGEYR'] >= 40) & (merged['RIDAGEYR'] <= 60)]
print(f'\nAge 40-60 subset: {len(age_match)} participants')
print(f'  Male: {len(age_match[age_match["RIAGENDR"]==1])}, Female: {len(age_match[age_match["RIAGENDR"]==2])}')

print('\n=== Age 40-60 reference values (all sexes) ===')
key_vars = [('LBXSCA', 'Calcium'), ('LBXSKSI', 'Potassium'), ('LBXSNASI', 'Sodium'), 
            ('LBXSCLSI', 'Chloride'), ('LBXSC3SI', 'CO2'),
            ('LBXHGB', 'Hemoglobin'), ('LBXHCT', 'Hematocrit'), ('LBXRBCSI', 'RBC'),
            ('LBXMCHSI', 'MCH'), ('LBXMC', 'MCHC'), ('LBXMCVSI', 'MCV')]
for code, label in key_vars:
    vals = age_match[code].dropna()
    print(f'  {label:<14} n={len(vals):4d}  mean={vals.mean():.2f}  sd={vals.std():.2f}  [P2.5={vals.quantile(0.025):.2f}, P97.5={vals.quantile(0.975):.2f}]')

# Also male-only (astronauts are predominantly male)
males_40_60 = age_match[age_match['RIAGENDR'] == 1]
print(f'\n=== Age 40-60 MALE reference values (n={len(males_40_60)}) ===')
for code, label in key_vars:
    vals = males_40_60[code].dropna()
    print(f'  {label:<14} n={len(vals):4d}  mean={vals.mean():.2f}  sd={vals.std():.2f}  [P2.5={vals.quantile(0.025):.2f}, P97.5={vals.quantile(0.975):.2f}]')

BMI-related columns: []
All DEMO columns: ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMQADFC', 'DMDBORN4', 'DMDCITZN', 'DMDYRSUS', 'DMDEDUC3', 'DMDEDUC2', 'DMDMARTL', 'RIDEXPRG', 'SIALANG']

Merged dataset: (6401, 18)
Age range: 12 - 80
Sex: {2.0: 3286, 1.0: 3115} (1=Male, 2=Female)

Age 40-60 subset: 1794 participants
  Male: 854, Female: 940

=== Age 40-60 reference values (all sexes) ===
  Calcium        n=1681  mean=9.24  sd=0.36  [P2.5=8.60, P97.5=10.00]
  Potassium      n=1680  mean=4.04  sd=0.35  [P2.5=3.40, P97.5=4.70]
  Sodium         n=1681  mean=140.22  sd=2.78  [P2.5=135.00, P97.5=145.00]
  Chloride       n=1681  mean=100.90  sd=2.86  [P2.5=95.00, P97.5=106.00]
  CO2            n=1681  mean=25.68  sd=2.38  [P2.5=21.00, P97.5=30.00]
  Hemoglobin     n=1719  mean=14.08  sd=1.58  [P2.5=10.70, P97.5=16.70]
  Hematocrit     n=1719  mean=41.92  sd=4.25  [P2.5=33.09, P97.5=49.60]
  RBC          

In [33]:
# Quick comparison: astronaut values vs NHANES male 40-60 reference
import pandas as pd, numpy as np

astro = pd.read_csv('/mnt/results/data/astronaut_minerals_combined.csv')

# NHANES male 40-60 reference stats
nhanes_ref = {
    'CALCIUM':         {'mean': 9.24, 'sd': 0.34, 'p025': 8.50, 'p975': 9.90},
    'POTASSIUM':       {'mean': 4.10, 'sd': 0.35, 'p025': 3.40, 'p975': 4.80},
    'SODIUM':          {'mean': 140.17, 'sd': 2.77, 'p025': 135.00, 'p975': 145.00},
    'CHLORIDE':        {'mean': 100.64, 'sd': 2.93, 'p025': 94.00, 'p975': 106.00},
    'CARBON DIOXIDE':  {'mean': 25.82, 'sd': 2.43, 'p025': 21.00, 'p975': 30.00},
    'HEMOGLOBIN':      {'mean': 14.99, 'sd': 1.27, 'p025': 12.04, 'p975': 17.30},
    'HEMATOCRIT':      {'mean': 44.35, 'sd': 3.52, 'p025': 36.64, 'p975': 51.23},
    'RED BLOOD CELL COUNT': {'mean': 5.01, 'sd': 0.48, 'p025': 4.07, 'p975': 6.02},
    'MCH':             {'mean': 29.99, 'sd': 2.26, 'p025': 24.30, 'p975': 33.86},
    'MCHC':            {'mean': 33.77, 'sd': 0.86, 'p025': 31.93, 'p975': 35.30},
    'MCV':             {'mean': 88.78, 'sd': 5.70, 'p025': 75.83, 'p975': 99.23},
}

print('Astronaut values vs NHANES Male 40-60 reference:')
print(f'{"Variable":<22} {"Astro mean":>10} {"Astro range":>16} {"NHANES mean":>12} {"NHANES P2.5-P97.5":>20} {"% outside ref":>14}')
print('-' * 100)
for var in nhanes_ref:
    if var in astro.columns:
        astro_vals = astro[var].dropna()
        ref = nhanes_ref[var]
        z = (astro_vals - ref['mean']) / ref['sd']
        outside = ((astro_vals < ref['p025']) | (astro_vals > ref['p975'])).sum()
        print(f'{var:<22} {astro_vals.mean():>10.2f} {astro_vals.min():.1f}-{astro_vals.max():.1f}     {ref["mean"]:>10.2f}   {ref["p025"]:.1f}-{ref["p975"]:.1f}          {outside}/{len(astro_vals)} ({100*outside/len(astro_vals):.0f}%)')

# Z-scores by timepoint
print('\n=== Mean z-scores by timepoint ===')
astro['timepoint'] = astro['timepoint']
tp_order = ['L-92', 'L-44', 'L-3', 'R+1', 'R+45', 'R+82', 'R+194']
for var in ['CALCIUM', 'POTASSIUM', 'SODIUM', 'CHLORIDE', 'CARBON DIOXIDE', 'HEMOGLOBIN', 'HEMATOCRIT', 'RED BLOOD CELL COUNT', 'MCH', 'MCHC', 'MCV']:
    if var in astro.columns:
        ref = nhanes_ref[var]
        z_scores = (astro[var] - ref['mean']) / ref['sd']
        astro[f'{var}_z'] = z_scores
        tp_means = []
        for tp in tp_order:
            tp_data = astro[astro['timepoint'] == tp]
            if len(tp_data) > 0:
                tp_means.append(f'{tp_data[f"{var}_z"].mean():+.2f}')
            else:
                tp_means.append('  -  ')
        print(f'  {var:<22} ' + '  '.join(tp_means))

Astronaut values vs NHANES Male 40-60 reference:
Variable               Astro mean      Astro range  NHANES mean    NHANES P2.5-P97.5  % outside ref
----------------------------------------------------------------------------------------------------
CALCIUM                      9.73 9.2-10.3           9.24   8.5-9.9          9/28 (32%)
POTASSIUM                    4.02 3.0-5.9           4.10   3.4-4.8          3/28 (11%)
SODIUM                     139.29 136.0-143.0         140.17   135.0-145.0          0/28 (0%)
CHLORIDE                   102.43 97.0-106.0         100.64   94.0-106.0          0/28 (0%)
CARBON DIOXIDE              23.25 19.0-28.0          25.82   21.0-30.0          4/28 (14%)
HEMOGLOBIN                  14.38 11.5-17.0          14.99   12.0-17.3          2/27 (7%)
HEMATOCRIT                  43.07 34.7-50.0          44.35   36.6-51.2          3/27 (11%)
RED BLOOD CELL COUNT         4.74 3.7-5.6           5.01   4.1-6.0          5/27 (19%)
MCH                         30

In [34]:
import urllib.request, os, pandas as pd, numpy as np

# ============================================================
# Step 1: Download & pool 3 NHANES cycles (2013-2018)
# ============================================================

cycles = {
    '2013-2014': {'year_dir': '2013', 'suffix': '_H'},
    '2015-2016': {'year_dir': '2015', 'suffix': '_I'},
    '2017-2018': {'year_dir': '2017', 'suffix': '_J'},
}

file_types = ['BIOPRO', 'CBC', 'DEMO']

os.makedirs('/workspace/nhanes_raw', exist_ok=True)

downloaded = {}
for cycle_name, info in cycles.items():
    suffix = info['suffix']
    year_dir = info['year_dir']
    for ftype in file_types:
        fname = f'{ftype}{suffix}'
        outpath = f'/workspace/nhanes_raw/{fname}.XPT'
        url = f'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year_dir}/DataFiles/{fname}.XPT'
        
        if os.path.exists(outpath) and os.path.getsize(outpath) > 100000:
            print(f'{fname}: already have {os.path.getsize(outpath)} bytes')
        else:
            print(f'Downloading {fname} from {url}...')
            try:
                req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=120) as resp:
                    data = resp.read()
                with open(outpath, 'wb') as f:
                    f.write(data)
                print(f'  Downloaded {len(data)} bytes')
            except Exception as e:
                print(f'  FAILED: {e}')
                continue
        downloaded[fname] = outpath

print(f'\nTotal files downloaded: {len(downloaded)}')

  Downloaded 2127760 bytes
  Downloaded 1586640 bytes


  Downloaded 3833200 bytes
  Downloaded 2056320 bytes


  Downloaded 1543440 bytes
  Downloaded 3756480 bytes
BIOPRO_J: already have 2106080 bytes
CBC_J: already have 1476320 bytes
DEMO_J: already have 3412720 bytes

Total files downloaded: 9


In [35]:
# Parse all XPT files and pool 3 cycles
all_merged = []

for cycle_name, info in cycles.items():
    suffix = info['suffix']
    print(f'Processing {cycle_name}...')
    
    # Parse demographics
    demo = pd.read_sas(f'/workspace/nhanes_raw/DEMO{suffix}.XPT', format='xport')
    # Parse biochemistry
    biopro = pd.read_sas(f'/workspace/nhanes_raw/BIOPRO{suffix}.XPT', format='xport')
    # Parse CBC
    cbc = pd.read_sas(f'/workspace/nhanes_raw/CBC{suffix}.XPT', format='xport')
    
    # Select columns we need
    demo_cols = ['SEQN', 'RIAGENDR', 'RIDAGEYR']
    biopro_cols = ['SEQN', 'LBXSCA', 'LBXSKSI', 'LBXSNASI', 'LBXSCLSI', 'LBXSC3SI']
    cbc_cols = ['SEQN', 'LBXHGB', 'LBXHCT', 'LBXRBCSI', 'LBXMCHSI', 'LBXMC', 'LBXMCVSI',
                'LBXWBCSI', 'LBXRDW', 'LBXPLTSI', 'LBXMPSI']
    
    # Merge
    m = demo[demo_cols].merge(biopro[biopro_cols], on='SEQN', how='inner')
    m = m.merge(cbc[cbc_cols], on='SEQN', how='inner')
    m['cycle'] = cycle_name
    print(f'  Merged: {m.shape}, age range: {m["RIDAGEYR"].min():.0f}-{m["RIDAGEYR"].max():.0f}')
    all_merged.append(m)

pooled = pd.concat(all_merged, ignore_index=True)
print(f'\nPooled dataset: {pooled.shape}')
print(f'Age range: {pooled["RIDAGEYR"].min():.0f}-{pooled["RIDAGEYR"].max():.0f}')
print(f'Sex: {pooled["RIAGENDR"].value_counts().to_dict()}')

# Filter to age 40-60, all sexes
ref_pop = pooled[(pooled['RIDAGEYR'] >= 40) & (pooled['RIDAGEYR'] <= 60)].copy()
print(f'\nAge 40-60 reference population: {len(ref_pop)}')
print(f'  Male: {len(ref_pop[ref_pop["RIAGENDR"]==1])}, Female: {len(ref_pop[ref_pop["RIAGENDR"]==2])}')
print(f'  By cycle: {ref_pop["cycle"].value_counts().to_dict()}')

Processing 2013-2014...
  Merged: (6979, 19), age range: 12-80
Processing 2015-2016...
  Merged: (6744, 19), age range: 12-80
Processing 2017-2018...
  Merged: (6401, 19), age range: 12-80

Pooled dataset: (20124, 19)
Age range: 12-80
Sex: {2.0: 10375, 1.0: 9749}

Age 40-60 reference population: 5748
  Male: 2713, Female: 3035
  By cycle: {'2013-2014': 2034, '2015-2016': 1920, '2017-2018': 1794}


In [36]:
# ============================================================
# Step 2: Compute NHANES reference statistics
# ============================================================

# Variable mapping: astronaut_name -> (nhanes_code, label, units)
var_map = {
    'CALCIUM':                ('LBXSCA',    'Total Calcium', 'mg/dL'),
    'POTASSIUM':              ('LBXSKSI',   'Potassium', 'mmol/L'),
    'SODIUM':                 ('LBXSNASI',  'Sodium', 'mmol/L'),
    'CHLORIDE':               ('LBXSCLSI',  'Chloride', 'mmol/L'),
    'CARBON DIOXIDE':         ('LBXSC3SI',  'Bicarbonate', 'mmol/L'),
    'HEMOGLOBIN':             ('LBXHGB',    'Hemoglobin', 'g/dL'),
    'HEMATOCRIT':             ('LBXHCT',    'Hematocrit', '%'),
    'RED BLOOD CELL COUNT':   ('LBXRBCSI',  'RBC count', 'M/uL'),
    'MCH':                    ('LBXMCHSI',  'Mean cell Hgb', 'pg'),
    'MCHC':                   ('LBXMC',     'Mean Cell Hgb Conc', 'g/dL'),
    'MCV':                    ('LBXMCVSI',  'Mean cell volume', 'fL'),
    'WHITE BLOOD CELL COUNT': ('LBXWBCSI',  'WBC count', '10^3/uL'),
    'RDW':                    ('LBXRDW',    'RBC distribution width', '%'),
    'PLATELET COUNT':         ('LBXPLTSI',  'Platelet count', '10^3/uL'),
    'MPV':                    ('LBXMPSI',   'Mean platelet volume', 'fL'),
}

percentiles = [1, 2.5, 5, 10, 25, 50, 75, 90, 95, 97.5, 99]
ref_rows = []

for astro_var, (nhanes_code, label, units) in var_map.items():
    for sex_label, sex_filter in [('All', None), ('Male', 1.0), ('Female', 2.0)]:
        if sex_filter is not None:
            subset = ref_pop[ref_pop['RIAGENDR'] == sex_filter]
        else:
            subset = ref_pop
        
        vals = subset[nhanes_code].dropna()
        n = len(vals)
        
        row = {
            'astronaut_variable': astro_var,
            'nhanes_code': nhanes_code,
            'label': label,
            'units': units,
            'sex': sex_label,
            'N': n,
            'mean': vals.mean(),
            'sd': vals.std(),
            'median': vals.median(),
        }
        for p in percentiles:
            row[f'P{p}'] = vals.quantile(p / 100)
        ref_rows.append(row)

ref_stats = pd.DataFrame(ref_rows)
print(f'Reference stats table: {ref_stats.shape}')
print(ref_stats[['astronaut_variable', 'sex', 'N', 'mean', 'sd', 'P2.5', 'P97.5']].to_string(index=False))

# Save to /mnt/results/data/
ref_stats.to_csv('/mnt/results/data/nhanes_reference_stats.csv', index=False)
print('\nSaved nhanes_reference_stats.csv')

Reference stats table: (45, 20)
    astronaut_variable    sex    N       mean        sd      P2.5   P97.5
               CALCIUM    All 5429   9.327022  0.362681   8.70000  10.000
               CALCIUM   Male 2552   9.338284  0.350164   8.70000  10.000
               CALCIUM Female 2877   9.317032  0.373210   8.60000  10.100
             POTASSIUM    All 5442   3.983988  0.344267   3.30025   4.700
             POTASSIUM   Male 2562   4.044215  0.338308   3.40000   4.800
             POTASSIUM Female 2880   3.930410  0.340705   3.30000   4.600
                SODIUM    All 5444 139.484754  2.487807 135.00000 144.000
                SODIUM   Male 2563 139.488880  2.445059 134.00000 144.000
                SODIUM Female 2881 139.481083  2.525648 135.00000 145.000
              CHLORIDE    All 5444 102.827333  3.127979  96.00000 108.000
              CHLORIDE   Male 2563 102.571596  3.091097  96.00000 108.000
              CHLORIDE Female 2881 103.054842  3.143519  97.00000 108.000
      


Saved nhanes_reference_stats.csv


In [37]:
# ============================================================
# Step 3: Compute astronaut z-scores & percentile ranks
# ============================================================

# Load astronaut data
astro_minerals = pd.read_csv('/mnt/results/data/astronaut_minerals_combined.csv')
astro_cbc = pd.read_csv('/mnt/results/data/astronaut_cbc_full.csv')

# Merge minerals + CBC on SUBJECT_ID + timepoint
astro = astro_minerals.merge(
    astro_cbc[['SUBJECT_ID', 'timepoint', 'WHITE BLOOD CELL COUNT', 'RDW', 'PLATELET COUNT', 'MPV']],
    on=['SUBJECT_ID', 'timepoint'], how='outer'
)
print(f'Astronaut merged data: {astro.shape}')
print(f'Columns: {list(astro.columns)}')

# Get reference stats for 'All' sex (primary reference)
ref_all = ref_stats[ref_stats['sex'] == 'All'].set_index('astronaut_variable')

# Compute z-scores and percentile ranks
zscore_cols = []
percentile_cols = []
outside_ref_cols = []

for astro_var, (nhanes_code, label, units) in var_map.items():
    if astro_var not in astro.columns:
        print(f'WARNING: {astro_var} not in astronaut data')
        continue
    
    ref_mean = ref_all.loc[astro_var, 'mean']
    ref_sd = ref_all.loc[astro_var, 'sd']
    ref_p025 = ref_all.loc[astro_var, 'P2.5']
    ref_p975 = ref_all.loc[astro_var, 'P97.5']
    
    # Z-score
    z_col = f'{astro_var}_z'
    astro[z_col] = (astro[astro_var] - ref_mean) / ref_sd
    zscore_cols.append(z_col)
    
    # Percentile rank (empirical CDF from NHANES reference)
    pct_col = f'{astro_var}_pctile'
    nhanes_vals = ref_pop[nhanes_code].dropna().values
    astro[pct_col] = astro[astro_var].apply(
        lambda x: np.nan if pd.isna(x) else (np.searchsorted(np.sort(nhanes_vals), x, side='right') / len(nhanes_vals) * 100)
    )
    percentile_cols.append(pct_col)
    
    # Outside reference interval flag
    outside_col = f'{astro_var}_outside_ref'
    astro[outside_col] = ((astro[astro_var] < ref_p025) | (astro[astro_var] > ref_p975)).astype(int)
    outside_ref_cols.append(outside_col)

# Save z-scores table
zscore_table = astro[['SUBJECT_ID', 'timepoint'] + zscore_cols]
zscore_table.to_csv('/mnt/results/data/astronaut_zscores.csv', index=False)
print(f'\nSaved astronaut_zscores.csv: {zscore_table.shape}')

# Save percentile ranks table
pctile_table = astro[['SUBJECT_ID', 'timepoint'] + percentile_cols]
pctile_table.to_csv('/mnt/results/data/astronaut_percentile_ranks.csv', index=False)
print(f'Saved astronaut_percentile_ranks.csv: {pctile_table.shape}')

# Quick summary: mean z-scores by timepoint
tp_order = ['L-92', 'L-44', 'L-3', 'R+1', 'R+45', 'R+82', 'R+194']
print('\n=== Mean z-scores by timepoint (All-sexes reference) ===')
header = f'{"Variable":<28} ' + '  '.join(f'{tp:>7}' for tp in tp_order)
print(header)
for astro_var in var_map:
    z_col = f'{astro_var}_z'
    if z_col in astro.columns:
        vals = []
        for tp in tp_order:
            tp_data = astro[astro['timepoint'] == tp]
            if len(tp_data) > 0 and tp_data[z_col].notna().any():
                vals.append(f'{tp_data[z_col].mean():>+7.2f}')
            else:
                vals.append(f'{"--":>7}')
        print(f'{astro_var:<28} ' + '  '.join(vals))

Astronaut merged data: (28, 17)
Columns: ['SUBJECT_ID', 'timepoint', 'CALCIUM', 'POTASSIUM', 'SODIUM', 'CHLORIDE', 'CARBON DIOXIDE', 'HEMOGLOBIN', 'HEMATOCRIT', 'RED BLOOD CELL COUNT', 'MCH', 'MCHC', 'MCV', 'WHITE BLOOD CELL COUNT', 'RDW', 'PLATELET COUNT', 'MPV']



Saved astronaut_zscores.csv: (28, 17)


Saved astronaut_percentile_ranks.csv: (28, 17)

=== Mean z-scores by timepoint (All-sexes reference) ===
Variable                        L-92     L-44      L-3      R+1     R+45     R+82    R+194
CALCIUM                        +1.30    +1.92    +0.89    +0.55    +0.68    +1.72    +0.68
POTASSIUM                      +0.34    -0.03    -0.90    -0.75    +1.79    -0.03    +0.26
SODIUM                         +0.21    +0.11    +0.61    -0.60    -0.50    +0.11    -0.50
CHLORIDE                       -0.02    -0.42    +0.06    +0.14    -0.66    -0.02    +0.06
CARBON DIOXIDE                 -1.24    -0.38    -0.38    -1.24    -0.49    -0.16    -1.78
HEMOGLOBIN                     +0.23    +0.42    +0.04    +0.27    +0.29    +0.31    +0.09
HEMATOCRIT                     +0.36    +0.50    +0.23    +0.23    +0.39    +0.34    +0.08
RED BLOOD CELL COUNT           -0.04    +0.00    -0.33    +0.13    +0.16    +0.12    +0.01
MCH                            +0.35    +0.55    +0.41    +0.05    +0.16    

In [38]:
# Check MPV values - likely unit mismatch
print('Astronaut MPV values:')
print(astro[['SUBJECT_ID', 'timepoint', 'MPV', 'MPV_z']].to_string(index=False))
print(f'\nNHANES MPV (LBXMPSI) summary:')
print(ref_pop['LBXMPSI'].describe())
print(f'\nAstronaut MPV range: {astro["MPV"].min()} - {astro["MPV"].max()}')
print(f'NHANES MPV range: {ref_pop["LBXMPSI"].min():.1f} - {ref_pop["LBXMPSI"].max():.1f}')

# Also check RDW - z-score of +2.71 at L-92 seems high but plausible
print('\nAstronaut RDW values:')
print(astro[['SUBJECT_ID', 'timepoint', 'RDW', 'RDW_z']].to_string(index=False))
print(f'NHANES RDW range: {ref_pop["LBXRDW"].min():.1f} - {ref_pop["LBXRDW"].max():.1f}')

Astronaut MPV values:
SUBJECT_ID timepoint   MPV      MPV_z
      C001       L-3  10.9   2.737981
      C001      L-44  10.6   2.414060
      C001      L-92  10.3   2.090139
      C001       R+1  10.8   2.630007
      C001     R+194  10.5   2.306086
      C001      R+45  10.5   2.306086
      C001      R+82  10.8   2.630007
      C002       L-3  10.8   2.630007
      C002      L-44  10.9   2.737981
      C002      L-92  10.1   1.874191
      C002       R+1  10.2   1.982165
      C002     R+194  10.8   2.630007
      C002      R+45  11.0   2.845955
      C002      R+82  10.5   2.306086
      C003       L-3  10.8   2.630007
      C003      L-44  10.3   2.090139
      C003      L-92 330.0 347.281990
      C003       R+1  10.5   2.306086
      C003     R+194  10.2   1.982165
      C003      R+45  10.4   2.198112
      C003      R+82  10.2   1.982165
      C004       L-3  10.1   1.874191
      C004      L-44   9.9   1.658244
      C004      L-92   9.5   1.226349
      C004       R+1  10.0  

In [39]:
# Check all C003 L-92 values and look for other outliers
print('C003 L-92 row:')
print(astro[astro['SUBJECT_ID'] == 'C003'][astro['timepoint'] == 'L-92'].T)

# Check for values outside plausible clinical ranges
clinical_ranges = {
    'CALCIUM': (6, 14), 'POTASSIUM': (2, 8), 'SODIUM': (120, 160),
    'CHLORIDE': (80, 120), 'CARBON DIOXIDE': (10, 40),
    'HEMOGLOBIN': (5, 22), 'HEMATOCRIT': (15, 65), 'RED BLOOD CELL COUNT': (2, 8),
    'MCH': (15, 45), 'MCHC': (25, 40), 'MCV': (50, 120),
    'WHITE BLOOD CELL COUNT': (1, 50), 'RDW': (10, 25),
    'PLATELET COUNT': (50, 600), 'MPV': (5, 15),
}

print('\n=== Outlier check (outside plausible clinical ranges) ===')
for var, (lo, hi) in clinical_ranges.items():
    if var in astro.columns:
        outliers = astro[(astro[var] < lo) | (astro[var] > hi)]
        if len(outliers) > 0:
            for _, row in outliers.iterrows():
                print(f'  {var}: {row[var]} at {row["SUBJECT_ID"]} {row["timepoint"]} (range: {lo}-{hi})')

# Set implausible values to NaN
print('\nSetting implausible values to NaN...')
for var, (lo, hi) in clinical_ranges.items():
    if var in astro.columns:
        mask = (astro[var] < lo) | (astro[var] > hi)
        n_flagged = mask.sum()
        if n_flagged > 0:
            astro.loc[mask, var] = np.nan
            print(f'  {var}: flagged {n_flagged} value(s)')

# Recompute z-scores and percentile ranks after cleaning
for astro_var, (nhanes_code, label, units) in var_map.items():
    if astro_var not in astro.columns:
        continue
    ref_mean = ref_all.loc[astro_var, 'mean']
    ref_sd = ref_all.loc[astro_var, 'sd']
    ref_p025 = ref_all.loc[astro_var, 'P2.5']
    ref_p975 = ref_all.loc[astro_var, 'P97.5']
    
    z_col = f'{astro_var}_z'
    astro[z_col] = (astro[astro_var] - ref_mean) / ref_sd
    
    pct_col = f'{astro_var}_pctile'
    nhanes_vals = ref_pop[nhanes_code].dropna().values
    astro[pct_col] = astro[astro_var].apply(
        lambda x: np.nan if pd.isna(x) else (np.searchsorted(np.sort(nhanes_vals), x, side='right') / len(nhanes_vals) * 100)
    )
    
    outside_col = f'{astro_var}_outside_ref'
    astro[outside_col] = ((astro[astro_var] < ref_p025) | (astro[astro_var] > ref_p975)).astype(int)
    astro.loc[astro[astro_var].isna(), outside_col] = np.nan

# Re-save cleaned tables
zscore_table = astro[['SUBJECT_ID', 'timepoint'] + [f'{v}_z' for v in var_map if f'{v}_z' in astro.columns]]
zscore_table.to_csv('/mnt/results/data/astronaut_zscores.csv', index=False)

pctile_table = astro[['SUBJECT_ID', 'timepoint'] + [f'{v}_pctile' for v in var_map if f'{v}_pctile' in astro.columns]]
pctile_table.to_csv('/mnt/results/data/astronaut_percentile_ranks.csv', index=False)

print(f'\nRe-saved cleaned zscores ({zscore_table.shape}) and percentile ranks ({pctile_table.shape})')

# Print cleaned z-score summary
print('\n=== Cleaned mean z-scores by timepoint ===')
header = f'{"Variable":<28} ' + '  '.join(f'{tp:>7}' for tp in tp_order)
print(header)
for astro_var in var_map:
    z_col = f'{astro_var}_z'
    if z_col in astro.columns:
        vals = []
        for tp in tp_order:
            tp_data = astro[astro['timepoint'] == tp]
            if len(tp_data) > 0 and tp_data[z_col].notna().any():
                vals.append(f'{tp_data[z_col].mean():>+7.2f}')
            else:
                vals.append(f'{"--":>7}')
        print(f'{astro_var:<28} ' + '  '.join(vals))

C003 L-92 row:
                                   16
SUBJECT_ID                       C003
timepoint                        L-92
CALCIUM                           9.3
POTASSIUM                         3.7
SODIUM                          139.0
...                               ...
PLATELET COUNT_pctile             0.0
PLATELET COUNT_outside_ref          1
MPV_z                       347.28199
MPV_pctile                      100.0
MPV_outside_ref                     1

[62 rows x 1 columns]

=== Outlier check (outside plausible clinical ranges) ===
  RDW: 32.4 at C003 L-92 (range: 10-25)
  PLATELET COUNT: 12.9 at C003 L-92 (range: 50-600)
  MPV: 330.0 at C003 L-92 (range: 5-15)

Setting implausible values to NaN...
  RDW: flagged 1 value(s)
  PLATELET COUNT: flagged 1 value(s)
  MPV: flagged 1 value(s)


/tmp/ipykernel_333/4071596831.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(astro[astro['SUBJECT_ID'] == 'C003'][astro['timepoint'] == 'L-92'].T)



Re-saved cleaned zscores ((28, 17)) and percentile ranks ((28, 17))

=== Cleaned mean z-scores by timepoint ===
Variable                        L-92     L-44      L-3      R+1     R+45     R+82    R+194
CALCIUM                        +1.30    +1.92    +0.89    +0.55    +0.68    +1.72    +0.68
POTASSIUM                      +0.34    -0.03    -0.90    -0.75    +1.79    -0.03    +0.26
SODIUM                         +0.21    +0.11    +0.61    -0.60    -0.50    +0.11    -0.50
CHLORIDE                       -0.02    -0.42    +0.06    +0.14    -0.66    -0.02    +0.06
CARBON DIOXIDE                 -1.24    -0.38    -0.38    -1.24    -0.49    -0.16    -1.78
HEMOGLOBIN                     +0.23    +0.42    +0.04    +0.27    +0.29    +0.31    +0.09
HEMATOCRIT                     +0.36    +0.50    +0.23    +0.23    +0.39    +0.34    +0.08
RED BLOOD CELL COUNT           -0.04    +0.00    -0.33    +0.13    +0.16    +0.12    +0.01
MCH                            +0.35    +0.55    +0.41    +0.05    +

In [40]:
# ============================================================
# Step 4: One-sample tests with FDR correction
# ============================================================
from scipy import stats

# For each variable × timepoint, test whether astronaut mean differs from NHANES population mean
# H0: astronaut_mean = NHANES_mean (one-sample t-test on astronaut values)
# Also compute Wilcoxon signed-rank test (non-parametric)
# Effect size: Cohen's d = (astronaut_mean - NHANES_mean) / NHANES_SD

test_rows = []

for astro_var, (nhanes_code, label, units) in var_map.items():
    if astro_var not in astro.columns:
        continue
    
    ref_mean = ref_all.loc[astro_var, 'mean']
    ref_sd = ref_all.loc[astro_var, 'sd']
    ref_p025 = ref_all.loc[astro_var, 'P2.5']
    ref_p975 = ref_all.loc[astro_var, 'P97.5']
    
    for tp in tp_order:
        tp_data = astro[astro['timepoint'] == tp][astro_var].dropna()
        n = len(tp_data)
        
        if n < 2:
            test_rows.append({
                'variable': astro_var, 'timepoint': tp, 'n': n,
                'astro_mean': tp_data.mean() if n == 1 else np.nan,
                'nhanes_mean': ref_mean, 'nhanes_sd': ref_sd,
                'cohen_d': np.nan, 't_stat': np.nan, 't_pvalue': np.nan,
                'wilcoxon_stat': np.nan, 'wilcoxon_pvalue': np.nan,
                'mean_z': np.nan, 'pct_outside_ref': np.nan,
            })
            continue
        
        astro_mean = tp_data.mean()
        astro_sd = tp_data.std()
        
        # One-sample t-test: are astronaut values different from NHANES mean?
        t_stat, t_p = stats.ttest_1samp(tp_data, ref_mean)
        
        # Wilcoxon signed-rank test (non-parametric)
        try:
            w_stat, w_p = stats.wilcoxon(tp_data - ref_mean)
        except ValueError:
            # Wilcoxon fails if all values are identical or all differences are 0
            w_stat, w_p = np.nan, np.nan
        
        # Cohen's d (using NHANES SD as the reference SD)
        cohen_d = (astro_mean - ref_mean) / ref_sd
        
        # Mean z-score
        mean_z = (astro_mean - ref_mean) / ref_sd
        
        # % outside reference interval
        outside = ((tp_data < ref_p025) | (tp_data > ref_p975)).sum()
        pct_outside = 100 * outside / n
        
        test_rows.append({
            'variable': astro_var, 'timepoint': tp, 'n': n,
            'astro_mean': astro_mean, 'astro_sd': astro_sd,
            'nhanes_mean': ref_mean, 'nhanes_sd': ref_sd,
            'cohen_d': cohen_d, 't_stat': t_stat, 't_pvalue': t_p,
            'wilcoxon_stat': w_stat, 'wilcoxon_pvalue': w_p,
            'mean_z': mean_z, 'pct_outside_ref': pct_outside,
        })

test_df = pd.DataFrame(test_rows)

# FDR correction (Benjamini-Hochberg) across all tests
from statsmodels.stats.multitest import multipletests

valid_mask = test_df['t_pvalue'].notna()
_, fdr_p, _, _ = multipletests(test_df.loc[valid_mask, 't_pvalue'], method='fdr_bh')
test_df.loc[valid_mask, 't_fdr'] = fdr_p

_, fdr_w, _, _ = multipletests(test_df.loc[valid_mask, 'wilcoxon_pvalue'].fillna(1.0), method='fdr_bh')
test_df.loc[valid_mask, 'wilcoxon_fdr'] = fdr_w

test_df.to_csv('/mnt/results/data/nhanes_onesample_tests.csv', index=False)
print(f'Saved nhanes_onesample_tests.csv: {test_df.shape}')

# Show significant results (FDR < 0.05)
sig = test_df[test_df['t_fdr'] < 0.05].sort_values('t_fdr')
print(f'\n=== Significant results (t-test FDR < 0.05): {len(sig)} ===')
if len(sig) > 0:
    print(sig[['variable', 'timepoint', 'n', 'astro_mean', 'nhanes_mean', 'cohen_d', 't_pvalue', 't_fdr', 'mean_z', 'pct_outside_ref']].to_string(index=False))
else:
    print('No significant results at FDR < 0.05')

# Show results with |Cohen's d| >= 0.8 (large effect) regardless of p-value
large_effect = test_df[test_df['cohen_d'].abs() >= 0.8].sort_values('cohen_d', key=abs, ascending=False)
print(f'\n=== Large effect sizes (|Cohen d| >= 0.8): {len(large_effect)} ===')
print(large_effect[['variable', 'timepoint', 'n', 'astro_mean', 'nhanes_mean', 'cohen_d', 't_pvalue', 't_fdr', 'mean_z']].to_string(index=False))

Saved nhanes_onesample_tests.csv: (105, 16)

=== Significant results (t-test FDR < 0.05): 6 ===
variable timepoint  n  astro_mean  nhanes_mean  cohen_d  t_pvalue    t_fdr   mean_z  pct_outside_ref
     MPV      R+82  4      10.475     8.364215 2.279093  0.000452 0.047490 2.279093             50.0
     MPV      L-44  4      10.425     8.364215 2.225106  0.002364 0.047838 2.225106             50.0
     MPV       R+1  4      10.375     8.364215 2.171119  0.001415 0.047838 2.171119             50.0
     MPV       L-3  4      10.650     8.364215 2.468047  0.001139 0.047838 2.468047             75.0
     MPV      R+45  4      10.450     8.364215 2.252099  0.002672 0.047838 2.252099             50.0
     MPV     R+194  4      10.325     8.364215 2.117132  0.002734 0.047838 2.117132             50.0

=== Large effect sizes (|Cohen d| >= 0.8): 22 ===
      variable timepoint  n  astro_mean  nhanes_mean   cohen_d  t_pvalue    t_fdr    mean_z
           MPV       L-3  4   10.650000     8.364215  

In [41]:
# ============================================================
# Step 5: Fig S6 - Trajectory plots with NHANES reference bands
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

rc_params = matplotlib.rcParams
rc_params['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
rc_params['svg.fonttype'] = 'none'
rc_params['figure.dpi'] = 150

# Phylo colorblind-friendly palette
crew_colors = {'C001': '#0279EE', 'C002': '#75A025', 'C003': '#FF9400', 'C004': '#FD9BED'}
mean_color = '#000000'
ref_band_color = '#ECE9E2'
ref_band_edge = '#CCCCCC'

tp_order = ['L-92', 'L-44', 'L-3', 'R+1', 'R+45', 'R+82', 'R+194']
tp_x = list(range(len(tp_order)))
tp_x_map = {tp: i for i, tp in enumerate(tp_order)}

# Variable display info
display_info = {
    'CALCIUM':                ('Calcium', 'mg/dL'),
    'POTASSIUM':              ('Potassium', 'mmol/L'),
    'SODIUM':                 ('Sodium', 'mmol/L'),
    'CHLORIDE':               ('Chloride', 'mmol/L'),
    'CARBON DIOXIDE':         ('Bicarbonate (CO$_2$)', 'mmol/L'),
    'HEMOGLOBIN':             ('Hemoglobin', 'g/dL'),
    'HEMATOCRIT':             ('Hematocrit', '%'),
    'RED BLOOD CELL COUNT':   ('RBC Count', 'M/uL'),
    'MCH':                    ('MCH', 'pg'),
    'MCHC':                   ('MCHC', 'g/dL'),
    'MCV':                    ('MCV', 'fL'),
    'WHITE BLOOD CELL COUNT': ('WBC Count', '10$^3$/uL'),
    'RDW':                    ('RDW', '%'),
    'PLATELET COUNT':         ('Platelet Count', '10$^3$/uL'),
    'MPV':                    ('MPV', 'fL'),
}

biochem_vars = ['CALCIUM', 'POTASSIUM', 'SODIUM', 'CHLORIDE', 'CARBON DIOXIDE']
hematology_vars = ['HEMOGLOBIN', 'HEMATOCRIT', 'RED BLOOD CELL COUNT', 'MCH', 'MCHC', 'MCV',
                   'WHITE BLOOD CELL COUNT', 'RDW', 'PLATELET COUNT', 'MPV']
all_vars_ordered = biochem_vars + hematology_vars

n_vars = len(all_vars_ordered)
n_cols = 5
n_rows = 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 12))
axes_flat = axes.flatten()

for idx, var in enumerate(all_vars_ordered):
    ax = axes_flat[idx]
    label, units = display_info[var]
    
    ref_mean = ref_all.loc[var, 'mean']
    ref_p025 = ref_all.loc[var, 'P2.5']
    ref_p975 = ref_all.loc[var, 'P97.5']
    ref_p05 = ref_all.loc[var, 'P5']
    ref_p95 = ref_all.loc[var, 'P95']
    
    # NHANES reference bands
    ax.axhspan(ref_p025, ref_p975, alpha=0.4, color=ref_band_color, edgecolor=ref_band_edge, linewidth=0.5, zorder=0)
    ax.axhspan(ref_p05, ref_p95, alpha=0.2, color='#E9ED4C', edgecolor='none', zorder=0)
    ax.axhline(ref_mean, color='#999999', linestyle='--', linewidth=0.8, zorder=1)
    
    # Individual astronaut trajectories
    for crew in ['C001', 'C002', 'C003', 'C004']:
        crew_data = astro[astro['SUBJECT_ID'] == crew].sort_values('timepoint')
        crew_x = [tp_x_map[tp] for tp in crew_data['timepoint'] if tp in tp_x_map]
        crew_y = [crew_data[crew_data['timepoint'] == tp][var].values[0] for tp in crew_data['timepoint'] if tp in tp_x_map]
        valid = [(x, y) for x, y in zip(crew_x, crew_y) if not np.isnan(y)]
        if valid:
            vx, vy = zip(*valid)
            ax.plot(vx, vy, 'o-', color=crew_colors[crew], markersize=4, linewidth=1.2, alpha=0.7, label=crew)
    
    # Mean trajectory
    mean_vals = []
    for tp in tp_order:
        tp_data = astro[astro['timepoint'] == tp][var].dropna()
        mean_vals.append(tp_data.mean() if len(tp_data) > 0 else np.nan)
    ax.plot(tp_x, mean_vals, 's--', color=mean_color, markersize=5, linewidth=2, label='Crew mean', zorder=5)
    
    # Flight phase shading
    ax.axvspan(tp_x_map['R+1'] - 0.3, tp_x_map['R+1'] + 0.3, alpha=0.08, color='#FF9400', zorder=0)
    
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylabel(units, fontsize=9)
    ax.set_xticks(tp_x)
    ax.set_xticklabels(tp_order, fontsize=7, rotation=45, ha='right')
    ax.tick_params(axis='y', labelsize=8)
    
    if idx == 0:
        ax.legend(fontsize=7, loc='best', framealpha=0.8)

# Hide unused subplots
for idx in range(n_vars, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle('Astronaut Mineral & CBC Trajectories with NHANES Population Reference Bands\n(Age 40-60, All Sexes, Pooled 2013-2018)', 
             fontsize=14, fontweight='bold', y=0.98)
fig.text(0.5, 0.955, 'Grey band: P2.5-P97.5 | Yellow band: P5-P95 | Dashed line: NHANES mean | Orange shading: flight (R+1)', 
         ha='center', fontsize=9, color='#666666')
plt.tight_layout(rect=[0, 0, 1, 0.94])

fig.savefig('/mnt/results/figures/fig_s6_nhanes_trajectories.svg', format='svg', bbox_inches='tight')
fig.savefig('/mnt/results/figures/fig_s6_nhanes_trajectories.png', format='png', dpi=200, bbox_inches='tight')
plt.close()
print('Saved fig_s6_nhanes_trajectories.svg and .png')

/tmp/ipykernel_333/1299199711.py:65: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  ax.axhspan(ref_p025, ref_p975, alpha=0.4, color=ref_band_color, edgecolor=ref_band_edge, linewidth=0.5, zorder=0)
/tmp/ipykernel_333/1299199711.py:66: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  ax.axhspan(ref_p05, ref_p95, alpha=0.2, color='#E9ED4C', edgecolor='none', zorder=0)
/tmp/ipykernel_333/1299199711.py:65: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  ax.axhspan(ref_p025, ref_p975, alpha=0.4, color=ref_band_color, edgecolor=ref_band_edge, linewidth=0.5, zorder=0)
/tmp/ipykernel_333/1299199711.py:66: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  ax.axhspan(ref_p05, ref_p95, alpha=0.2, color='#E9ED4C', edgecolor='none', zorder=0)
/tmp/ipykernel_333/1299199711.py:65: UserWarning: Setting the 'col

/tmp/ipykernel_333/1299199711.py:65: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  ax.axhspan(ref_p025, ref_p975, alpha=0.4, color=ref_band_color, edgecolor=ref_band_edge, linewidth=0.5, zorder=0)
/tmp/ipykernel_333/1299199711.py:66: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  ax.axhspan(ref_p05, ref_p95, alpha=0.2, color='#E9ED4C', edgecolor='none', zorder=0)
/tmp/ipykernel_333/1299199711.py:65: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  ax.axhspan(ref_p025, ref_p975, alpha=0.4, color=ref_band_color, edgecolor=ref_band_edge, linewidth=0.5, zorder=0)
/tmp/ipykernel_333/1299199711.py:66: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  ax.axhspan(ref_p05, ref_p95, alpha=0.2, color='#E9ED4C', edgecolor='none', zorder=0)
/tmp/ipykernel_333/1299199711.py:65: UserWarning: Setting the 'col

/tmp/ipykernel_333/1299199711.py:65: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  ax.axhspan(ref_p025, ref_p975, alpha=0.4, color=ref_band_color, edgecolor=ref_band_edge, linewidth=0.5, zorder=0)
/tmp/ipykernel_333/1299199711.py:66: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  ax.axhspan(ref_p05, ref_p95, alpha=0.2, color='#E9ED4C', edgecolor='none', zorder=0)


Saved fig_s6_nhanes_trajectories.svg and .png


In [42]:
# ============================================================
# Step 6: Fig S7 - Z-score heatmap
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

rc_params = matplotlib.rcParams
rc_params['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
rc_params['svg.fonttype'] = 'none'
rc_params['figure.dpi'] = 150

# Build z-score matrix: rows = variables, cols = timepoints
z_matrix = []
var_labels_heatmap = []
for var in all_vars_ordered:
    z_col = f'{var}_z'
    row = []
    for tp in tp_order:
        tp_data = astro[astro['timepoint'] == tp]
        if z_col in tp_data.columns and tp_data[z_col].notna().any():
            row.append(tp_data[z_col].mean())
        else:
            row.append(np.nan)
    z_matrix.append(row)
    label, _ = display_info[var]
    var_labels_heatmap.append(label)

z_arr = np.array(z_matrix)

# Diverging colormap: blue (low) - white (normal) - red (high)
cmap = plt.cm.RdBu_r
norm = mcolors.TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)

fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(z_arr, cmap=cmap, norm=norm, aspect='auto', interpolation='nearest')

# Annotations
for i in range(z_arr.shape[0]):
    for j in range(z_arr.shape[1]):
        val = z_arr[i, j]
        if np.isnan(val):
            ax.text(j, i, '--', ha='center', va='center', fontsize=8, color='#999999')
        else:
            # Bold for |z| >= 1.96, outline for |z| >= 2.58
            weight = 'bold' if abs(val) >= 1.96 else 'normal'
            color = 'white' if abs(val) >= 1.5 else 'black'
            ax.text(j, i, f'{val:+.2f}', ha='center', va='center', fontsize=8, 
                    fontweight=weight, color=color)

ax.set_xticks(range(len(tp_order)))
ax.set_xticklabels(tp_order, fontsize=10, rotation=45, ha='right')
ax.set_yticks(range(len(var_labels_heatmap)))
ax.set_yticklabels(var_labels_heatmap, fontsize=10)

# Add vertical line separating pre-flight from flight/post-flight
ax.axvline(x=2.5, color='#333333', linewidth=1.5, linestyle='-')
ax.axvline(x=3.5, color='#333333', linewidth=1.5, linestyle='-')

# Labels for phases
ax.text(1, -1.2, 'Pre-flight', ha='center', fontsize=9, color='#666666')
ax.text(3, -1.2, 'Flight', ha='center', fontsize=9, color='#FF9400', fontweight='bold')
ax.text(5, -1.2, 'Post-flight', ha='center', fontsize=9, color='#666666')

cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Mean Z-score (vs NHANES)', fontsize=10)

# Add reference lines on colorbar
cbar.ax.axhline(y=1.96, color='black', linewidth=0.8, linestyle='--')
cbar.ax.axhline(y=-1.96, color='black', linewidth=0.8, linestyle='--')

ax.set_title('Astronaut Z-Score Heatmap vs NHANES Population Reference\n(Age 40-60, All Sexes, Pooled 2013-2018)', 
             fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
fig.savefig('/mnt/results/figures/fig_s7_zscore_heatmap.svg', format='svg', bbox_inches='tight')
fig.savefig('/mnt/results/figures/fig_s7_zscore_heatmap.png', format='png', dpi=200, bbox_inches='tight')
plt.close()
print('Saved fig_s7_zscore_heatmap.svg and .png')

Saved fig_s7_zscore_heatmap.svg and .png


In [43]:
# ============================================================
# Step 7: Fig S8 - Percentile rank dot plot
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

rc_params = matplotlib.rcParams
rc_params['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
rc_params['svg.fonttype'] = 'none'
rc_params['figure.dpi'] = 150

# Timepoint colors (colorblind-friendly progression)
tp_colors = {
    'L-92':  '#0279EE',
    'L-44':  '#4A9FEE',
    'L-3':   '#75A025',
    'R+1':   '#FF9400',
    'R+45':  '#E97400',
    'R+82':  '#CC5500',
    'R+194': '#FD9BED',
}

fig, ax = plt.subplots(figsize=(14, 8))

var_x_map = {var: i for i, var in enumerate(all_vars_ordered)}

# Plot each astronaut-timepoint as a dot
for _, row in astro.iterrows():
    for var in all_vars_ordered:
        pct_col = f'{var}_pctile'
        if pct_col in row.index and not pd.isna(row[pct_col]):
            x = var_x_map[var] + (tp_x_map[row['timepoint']] - 3) * 0.08  # jitter by timepoint
            y = row[pct_col]
            ax.scatter(x, y, color=tp_colors[row['timepoint']], s=25, alpha=0.6, edgecolors='none', zorder=3)

# Add reference bands
ax.axhspan(2.5, 97.5, alpha=0.08, color='#ECE9E2', zorder=0)
ax.axhspan(5, 95, alpha=0.06, color='#E9ED4C', zorder=0)
ax.axhline(50, color='#999999', linestyle='--', linewidth=0.8, zorder=1)
ax.axhline(2.5, color='#CCCCCC', linestyle=':', linewidth=0.5, zorder=1)
ax.axhline(97.5, color='#CCCCCC', linestyle=':', linewidth=0.5, zorder=1)

# X-axis labels
ax.set_xticks(range(len(all_vars_ordered)))
ax.set_xticklabels([display_info[v][0] for v in all_vars_ordered], fontsize=9, rotation=45, ha='right')
ax.set_ylabel('NHANES Percentile Rank (%)', fontsize=11)
ax.set_ylim(-2, 102)
ax.set_xlim(-0.5, len(all_vars_ordered) - 0.5)

# Legend for timepoints
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor=tp_colors[tp], 
                          markersize=7, label=tp) for tp in tp_order]
ax.legend(handles=legend_elements, loc='upper left', fontsize=8, ncol=4, framealpha=0.8)

# Add vertical line separating biochemistry from hematology
ax.axvline(x=4.5, color='#333333', linewidth=0.8, linestyle='-', alpha=0.3)
ax.text(2, 101, 'Biochemistry', ha='center', fontsize=9, color='#666666')
ax.text(9.5, 101, 'Hematology', ha='center', fontsize=9, color='#666666')

ax.set_title('Astronaut Percentile Ranks vs NHANES Population Reference\n(Age 40-60, All Sexes, Pooled 2013-2018)', 
             fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
fig.savefig('/mnt/results/figures/fig_s8_percentile_ranks.svg', format='svg', bbox_inches='tight')
fig.savefig('/mnt/results/figures/fig_s8_percentile_ranks.png', format='png', dpi=200, bbox_inches='tight')
plt.close()
print('Saved fig_s8_percentile_ranks.svg and .png')

# Verify all figures
import os
for f in ['fig_s6_nhanes_trajectories', 'fig_s7_zscore_heatmap', 'fig_s8_percentile_ranks']:
    for ext in ['svg', 'png']:
        path = f'/mnt/results/figures/{f}.{ext}'
        size = os.path.getsize(path)
        print(f'  {f}.{ext}: {size:,} bytes')

Saved fig_s8_percentile_ranks.svg and .png
  fig_s6_nhanes_trajectories.svg: 234,709 bytes
  fig_s6_nhanes_trajectories.png: 1,157,430 bytes
  fig_s7_zscore_heatmap.svg: 55,046 bytes
  fig_s7_zscore_heatmap.png: 246,678 bytes
  fig_s8_percentile_ranks.svg: 306,383 bytes
  fig_s8_percentile_ranks.png: 269,665 bytes


In [44]:
# Pull outside-reference percentages
all_vars_ordered = ['CALCIUM', 'POTASSIUM', 'SODIUM', 'CHLORIDE', 'CARBON DIOXIDE',
    'HEMOGLOBIN', 'HEMATOCRIT', 'RED BLOOD CELL COUNT', 'MCH', 'MCHC', 'MCV',
    'WHITE BLOOD CELL COUNT', 'RDW', 'PLATELET COUNT', 'MPV']

print('Variables with highest % outside P2.5-P97.5 reference (averaged across timepoints):')
for var in all_vars_ordered:
    outside_col = f'{var}_outside_ref'
    if outside_col in astro.columns:
        avg_outside = astro[outside_col].mean() * 100
        print(f'  {var}: {avg_outside:.1f}%')

Variables with highest % outside P2.5-P97.5 reference (averaged across timepoints):
  CALCIUM: 17.9%
  POTASSIUM: 10.7%
  SODIUM: 0.0%
  CHLORIDE: 0.0%
  CARBON DIOXIDE: 14.3%
  HEMOGLOBIN: 3.7%
  HEMATOCRIT: 3.7%
  RED BLOOD CELL COUNT: 7.4%
  MCH: 0.0%
  MCHC: 4.2%
  MCV: 0.0%
  WHITE BLOOD CELL COUNT: 0.0%
  RDW: 18.5%
  PLATELET COUNT: 3.7%
  MPV: 48.1%


In [45]:
import pandas as pd
import numpy as np
import gzip

# ── 1. Load mineral-pathway gene sets ──────────────────────────────────
gmt_path = '/mnt/results/data/mineral_pathway_genesets.gmt'
mineral_genes = set()
with open(gmt_path) as f:
    for line in f:
        parts = line.strip().split('\t')
        mineral_genes.update(parts[2:])
print(f'Mineral-pathway genes in GMT: {len(mineral_genes)}')

# ── 2. Load I4 counts (Ensembl) and map to symbols ────────────────────
i4_counts = pd.read_csv('/mnt/results/data/astronaut_rnaseq_counts.csv', index_col=0)
print(f'I4 counts: {i4_counts.shape}')

# Load Ensembl→symbol mapping
ens_map = pd.read_csv('/mnt/results/data/ensembl_to_symbol.csv')
print(f'Ensembl map columns: {list(ens_map.columns)}')
print(ens_map.head(2))

# Create mapping dict (strip version suffix from Ensembl IDs in counts)
ens_map['ens_clean'] = ens_map.iloc[:,0].astype(str).str.split('.').str[0]
symbol_col = [c for c in ens_map.columns if 'symbol' in c.lower() or 'hgnc' in c.lower() or 'gene' in c.lower()]
print(f'Symbol column candidates: {symbol_col}')
if symbol_col:
    sym_col = symbol_col[0]
else:
    sym_col = ens_map.columns[1]
print(f'Using symbol column: {sym_col}')
ens_to_sym = dict(zip(ens_map['ens_clean'], ens_map[sym_col]))

# Map I4 index (strip version)
i4_idx_clean = i4_counts.index.str.split('.').str[0]
i4_symbols = [ens_to_sym.get(e, None) for e in i4_idx_clean]
i4_counts.index = i4_symbols
# Drop unmapped
i4_counts = i4_counts[i4_counts.index.notna()]
# Aggregate duplicate symbols (sum counts)
i4_counts = i4_counts.groupby(i4_counts.index).sum()
print(f'I4 after symbol mapping: {i4_counts.shape}')

# CPM normalize then log2
i4_cpm = i4_counts.div(i4_counts.sum(axis=0), axis=1) * 1e6
i4_log = np.log2(i4_cpm + 1)

# ── 3. Load AX-1 TPM (already gene symbols) ───────────────────────────
ax1 = pd.read_csv('/mnt/shared-workspace/shared/osdr_raw/OSD-903_GSE276490_TPM.tsv.gz', sep='\t', index_col=0)
print(f'AX-1 raw: {ax1.shape}')
print(f'AX-1 columns: {list(ax1.columns)}')
ax1_log = np.log2(ax1 + 1)

# ── 4. Intersect to common mineral-pathway genes ──────────────────────
i4_mineral = i4_log[i4_log.index.isin(mineral_genes)]
ax1_mineral = ax1_log[ax1_log.index.isin(mineral_genes)]
print(f'I4 mineral genes: {len(i4_mineral)}, AX-1 mineral genes: {len(ax1_mineral)}')

common_genes = sorted(set(i4_mineral.index) & set(ax1_mineral.index) & mineral_genes)
print(f'Common mineral-pathway genes: {len(common_genes)}')

i4_sel = i4_mineral.loc[common_genes]
ax1_sel = ax1_mineral.loc[common_genes]

# ── 5. Build unified metadata ─────────────────────────────────────────
# I4 samples: C001_L-92, etc. → astronaut=C001, timepoint=L-92, flight_status=pre
i4_meta = []
for col in i4_sel.columns:
    parts = col.split('_')
    astro = parts[0]
    tp = parts[1]
    # L-92, L-44, L-3 = pre-flight; R+1 = post-flight (immediate return)
    fs = 'pre' if tp.startswith('L') else 'post'
    i4_meta.append({
        'sample_id': col, 'study': 'I4', 'astronaut_id': astro,
        'timepoint': tp, 'flight_status': fs, 'tissue': 'whole_blood'
    })
i4_meta = pd.DataFrame(i4_meta)

# AX-1 samples: PS001_t0, PS001_t2, PS001_t3, PS004_t0, etc.
ax1_meta = []
for col in ax1_sel.columns:
    parts = col.split('_')
    astro = parts[0]
    tp = parts[1]
    # t0=pre-flight, t2=post-24h, t3=post-3months
    fs_map = {'t0': 'pre', 't2': 'post', 't3': 'post'}
    fs = fs_map.get(tp, 'unknown')
    ax1_meta.append({
        'sample_id': col, 'study': 'AX1', 'astronaut_id': astro,
        'timepoint': tp, 'flight_status': fs, 'tissue': 'whole_blood'
    })
ax1_meta = pd.DataFrame(ax1_meta)

metadata = pd.concat([i4_meta, ax1_meta], ignore_index=True)
print(f'\nUnified metadata ({len(metadata)} samples):')
print(metadata.to_string())
print(f'\nFlight status distribution:')
print(metadata['flight_status'].value_counts())

# ── 6. Combine expression matrices ────────────────────────────────────
combined = pd.concat([i4_sel, ax1_sel], axis=1)
print(f'\nCombined expression matrix: {combined.shape}')

# ── 7. Save ───────────────────────────────────────────────────────────
combined.to_csv('/mnt/results/data/harmonized_astronaut_expression.csv')
metadata.to_csv('/mnt/results/data/harmonized_astronaut_metadata.csv', index=False)
print(f'\nSaved harmonized expression: {combined.shape} → /mnt/results/data/harmonized_astronaut_expression.csv')
print(f'Saved metadata: {metadata.shape} → /mnt/results/data/harmonized_astronaut_metadata.csv')

# Summary stats
print(f'\n--- Summary ---')
print(f'Genes: {len(common_genes)}')
print(f'Samples: {combined.shape[1]}')
print(f'Astronauts: {metadata["astronaut_id"].nunique()} ({list(metadata["astronaut_id"].unique())})')
print(f'Studies: {metadata["study"].unique().tolist()}')
print(f'Pre-flight samples: {(metadata["flight_status"]=="pre").sum()}')
print(f'Post-flight samples: {(metadata["flight_status"]=="post").sum()}')
print(f'Value range: [{combined.min().min():.2f}, {combined.max().max():.2f}]')
print(f'Mean per sample: {combined.mean().round(2).to_dict()}')

Mineral-pathway genes in GMT: 801


I4 counts: (61852, 16)


Ensembl map columns: ['ensembl_id', 'gene_symbol', 'gene_name']
           ensembl_id gene_symbol      gene_name
0  ENSG00000000003.15      TSPAN6  tetraspanin 6
1   ENSG00000000005.6        TNMD    tenomodulin
Symbol column candidates: ['gene_symbol', 'gene_name']
Using symbol column: gene_symbol
I4 after symbol mapping: (44458, 16)


AX-1 raw: (57923, 6)
AX-1 columns: ['PS001_t0', 'PS001_t2', 'PS001_t3', 'PS004_t0', 'PS004_t2', 'PS004_t3']
I4 mineral genes: 784, AX-1 mineral genes: 779
Common mineral-pathway genes: 779

Unified metadata (22 samples):
    sample_id study astronaut_id timepoint flight_status       tissue
0   C001_L-92    I4         C001      L-92           pre  whole_blood
1   C001_L-44    I4         C001      L-44           pre  whole_blood
2    C001_L-3    I4         C001       L-3           pre  whole_blood
3    C001_R+1    I4         C001       R+1          post  whole_blood
4   C002_L-92    I4         C002      L-92           pre  whole_blood
5   C002_L-44    I4         C002      L-44           pre  whole_blood
6    C002_L-3    I4         C002       L-3           pre  whole_blood
7    C002_R+1    I4         C002       R+1          post  whole_blood
8   C003_L-92    I4         C003      L-92           pre  whole_blood
9   C003_L-44    I4         C003      L-44           pre  whole_blood
10   C003


Saved harmonized expression: (779, 22) → /mnt/results/data/harmonized_astronaut_expression.csv
Saved metadata: (22, 6) → /mnt/results/data/harmonized_astronaut_metadata.csv

--- Summary ---
Genes: 779
Samples: 22
Astronauts: 6 (['C001', 'C002', 'C003', 'C004', 'PS001', 'PS004'])
Studies: ['I4', 'AX1']
Pre-flight samples: 14
Post-flight samples: 8
Value range: [0.00, 13.14]
Mean per sample: {'C001_L-92': 1.16, 'C001_L-44': 1.5, 'C001_L-3': 1.68, 'C001_R+1': 1.52, 'C002_L-92': 0.86, 'C002_L-44': 1.25, 'C002_L-3': 1.59, 'C002_R+1': 1.09, 'C003_L-92': 1.14, 'C003_L-44': 1.41, 'C003_L-3': 1.38, 'C003_R+1': 1.88, 'C004_L-92': 1.14, 'C004_L-44': 1.33, 'C004_L-3': 1.27, 'C004_R+1': 1.81, 'PS001_t0': 1.59, 'PS001_t2': 1.73, 'PS001_t3': 1.69, 'PS004_t0': 1.32, 'PS004_t2': 1.51, 'PS004_t3': 1.3}


In [46]:
import pandas as pd
import numpy as np

# Load harmonized data
expr = pd.read_csv('/mnt/results/data/harmonized_astronaut_expression.csv', index_col=0)
meta = pd.read_csv('/mnt/results/data/harmonized_astronaut_metadata.csv')

print(f'Expression: {expr.shape}, Metadata: {meta.shape}')

# ── Batch correction: z-score within each study ───────────────────────
studies = meta['study'].unique().tolist()
expr_z = pd.DataFrame(index=expr.index, columns=expr.columns, dtype=float)
for study in studies:
    cols = meta.loc[meta['study'] == study, 'sample_id'].tolist()
    sub = expr[cols]
    mu = sub.mean(axis=1).to_numpy()
    sd = sub.std(axis=1).to_numpy().copy()
    sd[sd == 0] = 1.0
    z = (sub.to_numpy() - mu[:, None]) / sd[:, None]
    for j, c in enumerate(cols):
        expr_z[c] = z[:, j]

print('After z-score: mean per study ~0, std ~1')
for study in studies:
    cols = meta.loc[meta['study'] == study, 'sample_id'].tolist()
    vals = expr_z[cols].to_numpy()
    print(f'  {study}: mean={vals.mean():.4f}, std={vals.std():.4f}')

expr_z.to_csv('/mnt/results/data/harmonized_expression_zscore.csv')
print(f'Saved z-scored expression: {expr_z.shape}')

# Prepare arrays for autoencoder
X = expr_z.to_numpy().T  # samples × genes
y_flight = (meta['flight_status'] == 'post').astype(int).to_numpy()
y_astronaut = meta['astronaut_id'].to_numpy()
y_study = meta['study'].to_numpy()
sample_ids = meta['sample_id'].to_numpy()
gene_names = list(expr_z.index)

print(f'\nAutoencoder input: X={X.shape} (samples × genes)')
print(f'Flight labels: {np.bincount(y_flight)} (0=pre, 1=post)')
print(f'Astronauts: {np.unique(y_astronaut, return_counts=True)}')
print(f'Gene count: {len(gene_names)}')
print(f'X range: [{X.min():.3f}, {X.max():.3f}]')

Expression: (779, 22), Metadata: (22, 6)
After z-score: mean per study ~0, std ~1
  I4: mean=-0.0000, std=0.7616
  AX1: mean=0.0000, std=0.9052


Saved z-scored expression: (779, 22)

Autoencoder input: X=(22, 779) (samples × genes)
Flight labels: [14  8] (0=pre, 1=post)
Astronauts: (array(['C001', 'C002', 'C003', 'C004', 'PS001', 'PS004'], dtype=object), array([4, 4, 4, 4, 3, 3]))
Gene count: 779
X range: [-2.826, 3.750]


In [47]:
cd /workspace && python train_autoencoder.py 2>&1

Data: 22 samples × 779 genes


Astronauts: ['C001' 'C002' 'C003' 'C004' 'PS001' 'PS004']


Device: cpu


=== LOAO Cross-Validation (6 folds) ===


Fold 1/6: test=C001 (train=18, test=4)


...

LOAO recon losses: [0.10253782570362091, 0.10864697396755219, 0.08300900459289551, 0.09242302924394608, 0.10561148077249527, 0.11590999364852905]


Mean LOAO loss: 0.101356


Final model loss: 0.131137


Latent dim: 16


Gene input dim: 779


In [48]:
cd /workspace && python classify_latent.py 2>&1

Latent: (22, 16), Raw: (22, 779)


Labels: [14  8] (0=pre, 1=post)


Astronauts: ['C001' 'C002' 'C003' 'C004' 'PS001' 'PS004']


=== Classification on Latent Space (16 features) ===


LogReg:


...

Classifier      Latent AUC   Raw AUC      Latent p   Raw p     


LogReg          0.509        0.429        0.4670     0.7040    


SVM_rbf         0.188        0.482        0.9900     0.4000    


SVM_linear      0.362        0.402        0.7390     0.6680    


RandomForest    0.728        0.621        0.0150     0.1670    


In [49]:
cd /workspace && python regress_latent.py 2>&1

I4 latent samples: 16


Matched samples: 16


Missing values per target:


  CALCIUM: 0 missing, range [9.2, 10.3]


  POTASSIUM: 0 missing, range [3.0, 4.6]


...

=== Comparison: Latent vs Raw (best r per target) ===


  CALCIUM: latent RandomForest |r|=0.366 vs raw SVR_rbf |r|=0.561


  POTASSIUM: latent RandomForest |r|=0.748 vs raw SVR_rbf |r|=0.681


  SODIUM: latent SVR_rbf |r|=0.431 vs raw SVR_rbf |r|=0.481


  HEMOGLOBIN: latent SVR_rbf |r|=0.915 vs raw SVR_rbf |r|=0.882


In [50]:
cd /workspace && python jaxa6_validation.py 2>&1

JAXA6 DGE: 26844 genes


I4 DE: 71 entries, 50 unique genes


JAXA6 columns: ['Row-names', 'Symbol', 'logFC', 'adj-P-Val', 'data']


I4 columns: ['gene_symbol', 'omics', 'comparison', 'log2FC', 'pvalue', 'adj_pvalue', 'minerals']


I4 comparisons: <ArrowStringArray>


...

logFC correlation: r=0.208 (p=0.4397)


Fisher enrichment: OR=0.00 (p=1.0000)


Meta-significant genes (FDR<0.05): 1


ML features in JAXA6 DE: 1/60


Saved: jaxa6_concordance_table.csv, jaxa6_stouffer_meta.csv, jaxa6_validation_results.csv


In [51]:
cd /workspace && python make_dl_figures.py 2>&1

Saved UMAP figure


Saved ROC figure


Saved concordance figure


Saved regression scatter figure


Saved AUC bar chart


All figures saved to /mnt/results/figures/


Files: dl_umap_latent, dl_roc_curves, dl_jaxa6_concordance, dl_regression_scatter, dl_classification_auc_bar


In [52]:
import pandas as pd
import numpy as np

# Load proteomics DE
prot_de = pd.read_csv('/mnt/results/data/astronaut_proteomics_de.csv')
print(f'Proteomics DE: {len(prot_de)} entries, {prot_de["Gene"].nunique()} unique genes')
print(f'Comparisons: {prot_de["_sheet"].unique()}')

# Filter to I4-FP1 (flight vs pre, most comparable to RNA-seq DE)
prot_fp1 = prot_de[prot_de['_sheet'] == 'I4-FP1'].copy()
print(f'\nI4-FP1 proteomics DE: {len(prot_fp1)} genes')

# Significant proteomics DE genes (adj.P.Val < 0.05)
prot_sig = prot_fp1[prot_fp1['adj.P.Val'] < 0.05].copy()
print(f'Significant (adj.P < 0.05): {len(prot_sig)} genes')
print(f'Top 10 by |logFC|:')
prot_sig['abs_logFC'] = prot_sig['logFC'].abs()
print(prot_sig.nlargest(10, 'abs_logFC')[['Gene', 'logFC', 'adj.P.Val']].to_string())

# Load RNA-seq mineral-pathway genes (779)
expr_z = pd.read_csv('/mnt/results/data/harmonized_expression_zscore.csv', index_col=0)
rna_genes = set(expr_z.index)
print(f'\nRNA-seq mineral-pathway genes: {len(rna_genes)}')

# Intersect: multi-omics consensus genes
prot_sig_genes = set(prot_sig['Gene'].unique())
multi_omics_genes = rna_genes & prot_sig_genes
print(f'Multi-omics consensus (RNA-seq ∩ proteomics DE): {len(multi_omics_genes)} genes')
print(f'Genes: {sorted(multi_omics_genes)}')

# Also check with a relaxed threshold (p < 0.05, not adj.P)
prot_relaxed = prot_fp1[prot_fp1['P.Value'] < 0.05]
prot_relaxed_genes = set(prot_relaxed['Gene'].unique())
multi_omics_relaxed = rna_genes & prot_relaxed_genes
print(f'\nRelaxed multi-omics (p < 0.05): {len(multi_omics_relaxed)} genes')

# Use the relaxed threshold if strict gives too few genes
if len(multi_omics_genes) < 10:
    print(f'Strict threshold gives only {len(multi_omics_genes)} genes — using relaxed (p < 0.05)')
    final_filter_genes = sorted(multi_omics_relaxed)
    filter_threshold = 'p < 0.05'
else:
    final_filter_genes = sorted(multi_omics_genes)
    filter_threshold = 'adj.P < 0.05'

print(f'\nFinal multi-omics filter: {len(final_filter_genes)} genes ({filter_threshold})')

# Save the filtered expression matrix
expr_filtered = expr_z.loc[final_filter_genes]
expr_filtered.to_csv('/mnt/results/data/harmonized_expression_multiomics_filtered.csv')
print(f'Saved filtered expression: {expr_filtered.shape}')

# Save gene list
pd.DataFrame({'gene': final_filter_genes, 'filter_threshold': filter_threshold}).to_csv(
    '/mnt/results/data/multiomics_consensus_genes.csv', index=False)
print(f'Saved gene list')

# Show the proteomics logFC for these genes
prot_filter = prot_fp1[prot_fp1['Gene'].isin(final_filter_genes)][['Gene', 'logFC', 'adj.P.Val', 'P.Value']].sort_values('Gene')
print(f'\nProteomics DE for filtered genes:')
print(prot_filter.to_string())

Proteomics DE: 3285 entries, 1765 unique genes
Comparisons: <ArrowStringArray>
['I4-FP1', 'I4-FP2']
Length: 2, dtype: str

I4-FP1 proteomics DE: 1765 genes
Significant (adj.P < 0.05): 59 genes
Top 10 by |logFC|:
      Gene     logFC  adj.P.Val
49    CALU  2.661721   0.029963
47   TARS1  2.518741   0.028215
57   FGA.1 -2.436513   0.046908
16   FBLN5 -2.103208   0.005995
22   ICAM3 -1.942013   0.012650
33   ABCF2 -1.860203   0.016574
8     SNCA  1.793406   0.002570
7    PF4V1  1.755976   0.002534
10  COL4A2 -1.743200   0.005025
42    SKP1  1.734350   0.020232

RNA-seq mineral-pathway genes: 779
Multi-omics consensus (RNA-seq ∩ proteomics DE): 3 genes
Genes: ['APP', 'FXYD2', 'PDGFA']

Relaxed multi-omics (p < 0.05): 15 genes
Strict threshold gives only 3 genes — using relaxed (p < 0.05)

Final multi-omics filter: 15 genes (p < 0.05)


Saved filtered expression: (15, 22)


Saved gene list

Proteomics DE for filtered genes:
         Gene     logFC  adj.P.Val   P.Value
3         APP  1.320346   0.001148  0.000003
156   ATP6AP1 -0.930772   0.148783  0.013243
180  ATP6V0D1 -0.579514   0.170122  0.017534
63       CYBB -0.799565   0.052059  0.001888
43      FXYD2 -0.930624   0.023390  0.000583
90       GCLM  1.047790   0.102251  0.005272
259      GPX1  0.545398   0.223716  0.032955
73        LOX -0.720257   0.069590  0.002918
188     LOXL1 -0.463575   0.171408  0.018355
17      PDGFA  1.059941   0.006423  0.000066
307   S100A11 -0.469817   0.245743  0.042883
172   S100A12 -0.997097   0.164818  0.016155
170    S100A8 -0.531235   0.164730  0.015984
174    S100A9 -0.518937   0.169816  0.016837
103      SOD3 -0.440631   0.111515  0.006793


In [53]:
cd /workspace && python train_vae.py 2>&1

Data: 22 samples × 779 genes


Device: cpu


=== VAE LOAO Cross-Validation (6 folds) ===


Fold 1/6: test=C001 (train=18, test=4)


  Best VAE loss: 0.689585 (epoch 32)


...

Saved final VAE latent features: (22, 20)


=== Summary ===


LOAO VAE losses: ['0.6896', '0.4537', '0.6595', '0.4691', '0.4248', '0.4381']


Mean LOAO loss: 0.522475


Final model loss: 0.457981


In [54]:
cd /workspace && python train_transformer.py 2>&1

Data: 22 samples × 779 genes


Device: cpu


=== FT-Transformer LOAO Cross-Validation (6 folds) ===


Architecture: 779 gene tokens, embed_dim=16, 4 heads, FFN=32, latent=16, dropout=0.5


Total parameters: 117,707


...

  Best recon loss: 0.411788 (epoch 150)


Fold 5/6: test=PS001 (train=19, test=3)


  Best recon loss: 0.382923 (epoch 150)


Fold 6/6: test=PS004 (train=19, test=3)




[Execution timed out after 30 minutes. Your code took too long to complete.

Tips:
- Break down long-running operations into smaller steps
- Increase the timeout parameter if the operation genuinely needs more time]

In [55]:
cd /workspace && python eval_all_architectures.py 2>&1

Feature spaces:


  DAE_latent: (22, 16)


  VAE_latent: (22, 16)


  Transformer_latent: (22, 16)


  Raw_779: (22, 779)


...

  SODIUM/RandomForest: r=-0.245, R2=-0.254, p=0.4680


  HEMOGLOBIN/Ridge: r=-0.390, R2=-1.113, p=0.2820


  HEMOGLOBIN/SVR_linear: r=-0.579, R2=-0.693, p=0.3340


  HEMOGLOBIN/SVR_rbf: r=-0.915, R2=-0.617, p=0.0380




[Execution timed out after 60 minutes. Your code took too long to complete.

Tips:
- Break down long-running operations into smaller steps
- Increase the timeout parameter if the operation genuinely needs more time]

In [56]:
cd /workspace && python eval_classification.py 2>&1



[Execution timed out after 45 minutes. Your code took too long to complete.

Tips:
- Break down long-running operations into smaller steps
- Increase the timeout parameter if the operation genuinely needs more time]

In [57]:
cd /workspace && python make_extended_figures.py 2>&1

Saved AUC comparison bar chart


Saved UMAP comparison


Saved ROC overlay


Saved regression comparison bar chart


All figures saved to /mnt/results/figures/


In [58]:
cd /workspace && python eval_ensemble.py 2>&1

=== Concatenation Ensemble (32-dim: DAE + Transformer) ===


  Ensemble_concat/LogReg: AUC=0.420, p=0.7480


  Ensemble_concat/SVM_rbf: AUC=0.205, p=0.9940


  Ensemble_concat/SVM_linear: AUC=0.482, p=0.3220


  Ensemble_concat/RF: AUC=0.692*, p=0.0240


...

Ensemble_concat      LogReg               0.420   0.7480   


Ensemble_concat      SVM_rbf              0.205   0.9940   


Ensemble_concat      SVM_linear           0.482   0.3220   


Ensemble_concat      RF                   0.692   0.0240   *


Ensemble_avg         DAE_RF+TFL_SVMrbf    0.750   0.0080   *


In [59]:
cd /workspace && python make_ensemble_figures.py 2>&1

Saved latent correlation heatmap


Saved per-astronaut AUC breakdown


Saved updated AUC bar chart


Saved updated ROC overlay


All figures saved to /mnt/results/figures/


In [60]:
cd /workspace && python download_nhanes_iron.py 2>&1

  Trying alternate URL: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/FETIN_H.xpt... FAILED: HTTP Error 404: Not Found


=== Loading and merging data ===


--- Cycle H ---


  DEMO: 10175 rows


  BIOPRO: 6979 rows, cols: ['SEQN', 'LBXSIR']


...

  TIBC: N=1678, mean=324.6, SD=50.9, P2.5=237.9, P97.5=442.2


Appended 4 rows to nhanes_reference_stats.csv


=== Iron-CBC Correlations (age 40-60, all sexes) ===


Saved filtered reference data (5748 rows) to workspace


Done!


In [61]:
cd /workspace && python compute_iron_correlations.py 2>&1

=== Loading CBC files ===


--- Cycle H ---


  CBC: 9422 rows, cols: ['SEQN', 'LBXHGB', 'LBXHCT', 'LBXRBCSI', 'LBXMCHSI', 'LBXMC', 'LBXMCVSI']


  Merged: 6979 rows


--- Cycle I ---


...

Saved iron reference distributions panel


Saved iron-CBC correlation heatmap


Saved combined iron reference figure


All figures saved to /mnt/results/figures/


Done!
